In [8]:
# =============================================================================
# Environment configuration (must be set BEFORE importing JAX)
# =============================================================================
import os
import warnings

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"   # Prevent full VRAM preallocation
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".6"     # Limit VRAM usage
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"  # Use platform-native allocator

warnings.filterwarnings("ignore", category=FutureWarning)

# =============================================================================
# Standard library imports
# =============================================================================
import sys
import io
import pickle
import logging
from pathlib import Path
from functools import partial
from typing import Optional, Sequence

# =============================================================================
# Core scientific Python stack
# =============================================================================
import numpy as np
import pandas as pd
import regex
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate

# Pandas utilities
# import janitor
from pandas.api.types import CategoricalDtype

# =============================================================================
# Statistical utilities
# =============================================================================
from scipy.stats import (
    skew,
    kurtosis,
    normaltest,
    wilcoxon,
)
from sklearn.preprocessing import LabelEncoder

# =============================================================================
# Bayesian analysis & diagnostics
# =============================================================================
import arviz as az

# =============================================================================
# Factor analysis (frequentist utilities)
# =============================================================================
from factor_analyzer import FactorAnalyzer, Rotator

# =============================================================================
# JAX ecosystem
# =============================================================================
import jax
import jax.numpy as jnp
from jax import random, jit, lax
import jax.nn as jnn
from jax.nn import softplus
from jax.scipy.special import ndtr
from jax.scipy.stats import norm
import jax.scipy.stats.norm as jnorm

# =============================================================================
# NumPyro probabilistic programming
# =============================================================================
import numpyro
import numpyro.distributions as dist
from numpyro import sample, deterministic, factor
from numpyro.infer import MCMC, NUTS, Predictive
from numpyro.infer.initialization import (
    init_to_value,
    init_to_median,
)
from numpyro.distributions import constraints
from numpyro.distributions.transforms import OrderedTransform
from numpyro.diagnostics import print_summary
from numpyro.infer import init_to_median
from numpyro.distributions import Distribution

# =============================================================================
# R interoperability (keep optional / notebook-only)
# =============================================================================
# import rpy2.robjects as robjects
# from rpy2.robjects import pandas2ri
# from rpy2.robjects.conversion import localconverter

# =============================================================================
# Automated EDA (optional / heavy dependency)
# =============================================================================
# from ydata_profiling import ProfileReport

# =============================================================================
# Pandas display configuration (not for library code)
# =============================================================================
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

All Bayesian models were estimated using NumPyro with the JAX backend (GPU acceleration when available). Runtime configuration and device information were logged programmatically to ensure reproducibility across environments.

In [9]:
from bayes_runtime import (
    configure_logging,
    log_jax_runtime_info,
    log_numpyro_runtime_info,
    log_slurm_info,
)

configure_logging()
log_slurm_info()
log_jax_runtime_info()
log_numpyro_runtime_info()

ModuleNotFoundError: No module named 'bayes_runtime'

In [10]:
# Base directory for data files
DATA_DIR = Path("/mnt/c/Users/rodan/OneDrive - Høgskulen på Vestlandet/Dokumenter/PhD/RQ2/data") # WSL

# DATA_DIR = Path("C:/Users/rodan/OneDrive - Høgskulen på Vestlandet/Dokumenter/PhD/RQ2/data") # PC
# Paths to cleaned datasets
PREOBS_FILE_PATH = DATA_DIR / "preobs_cleaned_qc_utf8.csv"
PILOT_FILE_PATH = DATA_DIR / "pilot_preobs_attr_cleaned_qc_utf8.csv"

In [ ]:
logger = logging.getLogger(__name__)

def read_csv_file(path: Path) -> pd.DataFrame:
    try:
        df = pd.read_csv(path, encoding="utf-8")
        logger.info(f"Successfully loaded data from {path}")
        return df
    except FileNotFoundError:
        logger.error(f"File not found: {path}")
        raise
    except pd.errors.ParserError as e:
        logger.error(f"Parsing error in file {path}: {e}")
        raise
    except Exception as e:
        logger.error(f"Unexpected error reading {path}: {e}")
        raise

df_preobs = read_csv_file(PREOBS_FILE_PATH)
df_pilot = read_csv_file(PILOT_FILE_PATH)

In [ ]:
# Define latent attributions mapping to item columns (success/failure valence)
latent_map = {
    "eff_1": ("amp_1", "amn_1"),
    "eff_2": ("amp_7", "amn_7"),
    "eff_3": ("amp_15", "amn_15"),
    "ab_1": ("amp_2", "amn_2"),
    "ab_2": ("amp_5", "amn_5"),
    "ab_3": ("amp_9", "amn_9"),
    "luck_1": ("amp_3", "amn_3"),
    "luck_2": ("amp_11", "amn_11"),
    "luck_3": ("amp_14", "amn_14"),
    "task_1": ("amp_4", "amn_4"),
    "task_2": ("amp_8", "amn_8"),
    "task_3": ("amp_12", "amn_12"),
    "int_1": ("amp_6", "amn_6"),
    "int_2": ("amp_10", "amn_10"),
    "int_3": ("amp_13", "amn_13"),
}

# Coalesce success ('amp_*') and failure ('amn_*') columns into unified attribution columns
for new_col, (amp_col, amn_col) in latent_map.items():
    df_preobs[new_col] = df_preobs[amp_col].combine_first(df_preobs[amn_col])
    df_pilot[new_col] = df_pilot[amp_col].combine_first(df_pilot[amn_col])
    # Drop original columns to avoid redundancy
    df_preobs.drop(columns=[amp_col, amn_col], inplace=True)
    df_pilot.drop(columns=[amp_col, amn_col], inplace=True)

# Add valence category based on scoring in preobs dataset
def assign_valence_preobs(score):
    if score in [3, 4]:
        return "success"
    elif score in [1, 2]:
        return "failure"
    else:
        return None

df_preobs["valence"] = df_preobs["sc"].apply(assign_valence_preobs).astype("category")

# Add valence category based on scoring in pilot dataset
def assign_valence_pilot(score):
    if score in ["Ganske godt", "Svært godt"]:
        return "success"
    elif score in ["Ganske dårleg", "Svært dårleg"]:
        return "failure"
    else:
        return None

df_pilot["valence"] = df_pilot["sc"].apply(assign_valence_pilot).astype("category")

# Drop rows with missing values in any of the latent attribution columns
df_preobs.dropna(subset=list(latent_map.keys()), inplace=True)
df_pilot.dropna(subset=list(latent_map.keys()), inplace=True)

# Filter out patterned responses based on flags in both datasets
df_preobs = df_preobs[~((df_preobs["attr_patterned"] == 1) & (df_preobs["total_patterned"] > 2))]
df_pilot = df_pilot[~((df_pilot["attr_patterned"] == 1) | (df_pilot["time_flag"] == 1))]

# Reorder columns: keep identifiers and valence first, then latent attribution columns
columns_order = ["id_unique", "valence"] + list(latent_map.keys())
preobs_attr_data = df_preobs[columns_order].copy()
pilot_attr_data = df_pilot[columns_order].copy()

In [ ]:
# Extract attribution item columns matching pattern *_[1-3]
item_cols = preobs_attr_data.filter(regex=r'_([1-3])$', axis=1).columns.tolist()

# Convert attribution columns to nullable integer type for both datasets
preobs_attr_data.loc[:, item_cols] = preobs_attr_data.loc[:, item_cols].astype('Int64')
pilot_attr_data.loc[:, item_cols] = pilot_attr_data.loc[:, item_cols].astype('Int64')

# Separate success and failure subsets from preobs dataset
preobs_success = preobs_attr_data.loc[preobs_attr_data['valence'] == 'success', item_cols]
preobs_failure = preobs_attr_data.loc[preobs_attr_data['valence'] == 'failure', item_cols]

# Separate failure subset from pilot dataset (success subset commented out)
pilot_success = pilot_attr_data.loc[pilot_attr_data['valence'] == 'success', item_cols]
pilot_failure = pilot_attr_data.loc[pilot_attr_data['valence'] == 'failure', item_cols]

# Define ordered categorical dtype with categories 1 through 6
ordered_cat_type = pd.CategoricalDtype(categories=list(range(1, 7)), ordered=True)

# Convert attribution columns in main datasets to ordered categorical
preobs_attr_data.loc[:, item_cols] = preobs_attr_data.loc[:, item_cols].astype(ordered_cat_type)
pilot_attr_data.loc[:, item_cols] = pilot_attr_data.loc[:, item_cols].astype(ordered_cat_type)

# Convert separated success/failure subsets to ordered categorical dtype
preobs_success = preobs_success.astype(ordered_cat_type)
preobs_failure = preobs_failure.astype(ordered_cat_type)

pilot_success = pilot_success.astype(ordered_cat_type)
pilot_failure = pilot_failure.astype(ordered_cat_type)

In [ ]:
# Convert preobs attribution data to JAX arrays (zero-indexed)
preobs_data_array = jnp.asarray(preobs_attr_data[item_cols].to_numpy(), dtype=jnp.int32) - 1

# Convert preobs success/failure subsets to JAX arrays (zero-indexed)
preobs_success_array = jnp.asarray(preobs_success.to_numpy(), dtype=jnp.int32) - 1
preobs_failure_array = jnp.asarray(preobs_failure.to_numpy(), dtype=jnp.int32) - 1

# Convert pilot attribution data to JAX arrays (zero-indexed)
pilot_data_array = jnp.asarray(pilot_attr_data[item_cols].to_numpy(), dtype=jnp.int32) - 1

# Convert pilot success/failure subsets to JAX arrays (zero-indexed)
pilot_success_array = jnp.asarray(pilot_success.to_numpy(), dtype=jnp.int32) - 1
pilot_failure_array = jnp.asarray(pilot_failure.to_numpy(), dtype=jnp.int32) - 1

In [ ]:
# Initialize LabelEncoder to convert text labels into numeric codes
le = LabelEncoder()

# Encode 'valence' in preobs dataset and store as int32
preobs_valence = le.fit_transform(preobs_attr_data['valence']).astype(np.int32)
preobs_attr_data.loc[:, 'valence_encoded'] = preobs_valence

# Encode 'valence' in pilot dataset using the same encoder to keep labels consistent
pilot_valence = le.transform(pilot_attr_data['valence']).astype(np.int32)
pilot_attr_data.loc[:, 'valence_encoded'] = pilot_valence

def create_group_indices(group_labels: np.ndarray) -> list[np.ndarray]:
    """Create list of index arrays, one per unique group label."""
    unique_groups = np.unique(group_labels)
    return [np.where(group_labels == g)[0] for g in unique_groups]

# Create index lists for preobs and pilot valence groups
preobs_valence_indices = create_group_indices(preobs_valence)
pilot_valence_indices = create_group_indices(pilot_valence)

# Print mapping of encoded groups to original labels
print("Group labels:", le.classes_)

In [ ]:
# Categories (e.g., 1 to 6)
categories = range(1, 7)

# Set seaborn style
sns.set(style="whitegrid")

# 1. Frequency distribution barplots per item
for col in item_cols:
    counts = preobs_attr_data[col].value_counts().reindex(categories, fill_value=0)
    plt.figure(figsize=(8, 4))
    sns.barplot(x=list(categories), y=counts.values, palette="muted")
    plt.title(f"Frequency Distribution: {col}")
    plt.xlabel("Category")
    plt.ylabel("Count")
    plt.xticks(ticks=list(categories), labels=list(categories))
    plt.show()

# 2. Stacked bar plot of category proportions across all items
prop_df = pd.DataFrame({
    col: preobs_attr_data[col].value_counts(normalize=True).reindex(categories, fill_value=0)
    for col in item_cols
}).T

prop_df.plot(kind='bar', stacked=True, colormap='Spectral', figsize=(12,6))
plt.title("Stacked Bar Plot of Category Proportions by Item")
plt.xlabel("Items")
plt.ylabel("Proportion")
plt.legend(title="Category", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# 3. Cumulative distribution function (CDF) plots per item
for col in item_cols:
    data = preobs_attr_data[col].dropna().astype(int)
    cdf = data.value_counts(normalize=True).sort_index().cumsum()
    plt.figure(figsize=(8, 4))
    sns.lineplot(x=cdf.index, y=cdf.values, marker="o")
    plt.title(f"Cumulative Distribution (CDF) of {col}")
    plt.xlabel("Category")
    plt.ylabel("Cumulative Proportion")
    plt.xticks(ticks=list(categories), labels=list(categories))
    plt.ylim(0, 1)
    plt.grid(True)
    plt.show()

# 4. Median and mode per item (descriptive summary)
summary_stats = []
for col in item_cols:
    data = preobs_attr_data[col].dropna().astype(int)
    median_val = data.median()
    mode_val = data.mode().iloc[0] if not data.mode().empty else np.nan
    summary_stats.append({
        "Item": col,
        "Median": median_val,
        "Mode": mode_val,
        "Count": len(data)
    })

summary_df = pd.DataFrame(summary_stats)
print(summary_df)

# 5. Nonparametric group comparisons (example: valence groups)
import scipy.stats as stats

# Assuming 'valence' is categorical with two groups: 'success' and 'failure'
group1 = preobs_attr_data[preobs_attr_data['valence'] == 'success']
group2 = preobs_attr_data[preobs_attr_data['valence'] == 'failure']

group_comp_results = []
for col in item_cols:
    data1 = group1[col].dropna().astype(int)
    data2 = group2[col].dropna().astype(int)
    # Mann-Whitney U test (nonparametric test for independent samples)
    if len(data1) > 0 and len(data2) > 0:
        stat, p_val = stats.mannwhitneyu(data1, data2, alternative='two-sided')
    else:
        stat, p_val = np.nan, np.nan
    group_comp_results.append({
        "Item": col,
        "Mann-Whitney U stat": stat,
        "p-value": p_val
    })

group_comp_df = pd.DataFrame(group_comp_results)
print(group_comp_df)

# Optional: adjust p-values for multiple testing (e.g., Benjamini-Hochberg FDR)
from statsmodels.stats.multitest import multipletests
pvals = group_comp_df["p-value"].values
_, pvals_adj, _, _ = multipletests(pvals, method='fdr_bh')
group_comp_df["p-value_adj"] = pvals_adj

print(group_comp_df)

Barplots of counts per category per item (best for ordinal)

Stacked bar plots to see proportions of categories across all items together

CDF line plots to visualize cumulative probabilities and compare how responses accumulate by category

Median and mode as robust central tendency for ordinal scales

Mann-Whitney U test comparing groups on each item (nonparametric test suitable for ordinal independent samples)

Multiple testing correction for group comparison p-values

# Bayesian EFA

In [6]:
# ============================================================
# Ordered Probit + Bayesian Ordinal Factor Model (EFA / CFA)
# - OrderedProbit: stable log Φ-diff with safe indexing + robust broadcasting
# - CFA: fixed marker loading = 1.0; positive primary loadings by default
# - EFA: standard identified block-triangular loadings (top-left k×k), diag > 0,
#        remaining rows free; optional oblique (LKJ) vs orthogonal factors
# - Thresholds: first finite threshold fixed to 0; strictly increasing via softplus+cumsum
# ============================================================

from __future__ import annotations

import numpy as np
import jax
import jax.numpy as jnp
from jax.nn import softplus
from jax.scipy.special import log_ndtr

import numpyro
import numpyro.distributions as dist
from numpyro.distributions import Distribution, constraints


def _logdiffexp(a: jnp.ndarray, b: jnp.ndarray) -> jnp.ndarray:
    """
    Stable log(exp(a) - exp(b)) assuming a >= b, with numerical clamp.
    """
    x = b - a
    x = jnp.minimum(x, 0.0)  # ensure exp(x) <= 1
    return a + jnp.log1p(-jnp.exp(x))


class OrderedProbit(Distribution):
    """
    Ordered probit distribution for integer categories 0..K-1 where K = n_thr + 1.

    mu: batch_shape (e.g., [n_obs, n_items])
    thresholds: broadcastable to mu.shape + (n_thr,)
                (e.g., [n_items, n_thr] or [n_obs, n_items, n_thr])
    """

    arg_constraints = {"mu": constraints.real, "thresholds": constraints.ordered_vector}
    support = constraints.nonnegative_integer

    def __init__(self, mu, thresholds, validate_args=None):
        self.mu = jnp.asarray(mu)
        thr = jnp.asarray(thresholds)

        if thr.ndim < 1:
            raise ValueError("thresholds must have at least 1 dimension (last dim = n_thr).")

        self.n_thr = int(thr.shape[-1])
        self.n_cats = self.n_thr + 1

        # Add leading singleton dims until thr has rank mu.ndim+1, then broadcast.
        while thr.ndim < self.mu.ndim + 1:
            thr = thr[None, ...]
        self.thresholds = jnp.broadcast_to(thr, self.mu.shape + (self.n_thr,))

        super().__init__(batch_shape=self.mu.shape, event_shape=(), validate_args=validate_args)

    def log_prob(self, value):
        y = jnp.asarray(value).astype(jnp.int32)
        in_range = (y >= 0) & (y < self.n_cats)

        # Clip for safe indexing (mask handles invalid values).
        y_idx = jnp.clip(y, 0, self.n_cats - 1)
        y_exp = y_idx[..., None]

        # Shift thresholds by mu: t_k - mu
        t = self.thresholds - self.mu[..., None]

        # Pad with -inf and +inf
        neg_inf = jnp.full(self.mu.shape + (1,), -jnp.inf)
        pos_inf = jnp.full(self.mu.shape + (1,),  jnp.inf)
        t_full = jnp.concatenate([neg_inf, t, pos_inf], axis=-1)  # [batch, n_cats+1]

        lower = jnp.take_along_axis(t_full[..., :-1], y_exp, axis=-1)[..., 0]
        upper = jnp.take_along_axis(t_full[...,  1:], y_exp, axis=-1)[..., 0]

        ordered = upper >= lower

        log_cdf_u = log_ndtr(upper)
        log_cdf_l = log_ndtr(lower)

        # Roundoff guard: for ordered=True, log_cdf_u should be >= log_cdf_l
        a = jnp.maximum(log_cdf_u, log_cdf_l)
        log_p = _logdiffexp(a, log_cdf_l)

        return jnp.where(in_range & ordered, log_p, -jnp.inf)

    def sample(self, key, sample_shape=()):
        shape = sample_shape + self.batch_shape
        z = self.mu + jax.random.normal(key, shape)
        y = jnp.sum(z[..., None] > self.thresholds, axis=-1)
        return y.astype(jnp.int32)


class BayesianOrdinalFactorModel:
    """
    Ordinal EFA/CFA with ordered probit likelihood.

    Defaults are geared toward robust inference in small/moderate samples:
    - CFA markers fixed to 1.0
    - CFA primary loadings positive by default (disable via allow_negative_cfa_loadings=True)
    - EFA identified via top-left k×k lower-triangular + positive diagonal
    - Optional oblique factors via LKJ Cholesky; orthogonal if efa_oblique=False
    """

    def __init__(
        self,
        n_factors: int,
        n_cats: int,
        mode: str = "efa",
        item_to_factor_indices=None,
        # Loadings priors
        loadings_scale: float = 0.5,
        cfa_loading_loc: float = 0.7,
        # Threshold priors
        threshold_offset: float = 0.1,
        delta_sd: float = 0.2,
        # Factor correlation prior (oblique EFA / correlated factors)
        lkj_eta: float = 2.0,
        efa_oblique: bool = True,
        # CFA sign convention
        allow_negative_cfa_loadings: bool = False,
        # Validation
        validate_cfa_coverage: bool = True,
    ):
        self.n_factors = int(n_factors)
        self.n_cats = int(n_cats)
        self.mode = mode.lower()
        self.item_to_factor_indices = item_to_factor_indices

        self.loadings_scale = float(loadings_scale)
        self.cfa_loading_loc = float(cfa_loading_loc)

        self.threshold_offset = float(threshold_offset)
        self.delta_sd = float(delta_sd)

        self.lkj_eta = float(lkj_eta)
        self.efa_oblique = bool(efa_oblique)

        self.allow_negative_cfa_loadings = bool(allow_negative_cfa_loadings)
        self.validate_cfa_coverage = bool(validate_cfa_coverage)

        if self.mode not in ("efa", "cfa"):
            raise ValueError("mode must be 'efa' or 'cfa'")

        if self.mode == "cfa" and self.item_to_factor_indices is None:
            raise ValueError("CFA mode requires item_to_factor_indices")

        # Precompute EFA top-left k×k strict-lower indices (static, not traced)
        r_lt, c_lt = np.tril_indices(self.n_factors, k=-1)
        self._efa_r_lt = jnp.asarray(r_lt, dtype=jnp.int32)
        self._efa_c_lt = jnp.asarray(c_lt, dtype=jnp.int32)
        self._efa_n_free_lt = int(r_lt.shape[0])

    def model(self, data):
        y = jnp.asarray(data).astype(jnp.int32)
        n_obs, n_items = y.shape
        k = self.n_factors

        if self.mode == "efa" and n_items < k:
            raise ValueError(f"EFA requires n_items ({n_items}) >= n_factors ({k}).")

        if self.mode == "cfa" and self.validate_cfa_coverage:
            # Host-side check; repeated per call but safe because uses Python sets on small lists
            all_assigned = set(i for idx in self.item_to_factor_indices for i in idx)
            missing = set(range(n_items)) - all_assigned
            if missing:
                raise ValueError(f"CFA: items not assigned to any factor: {sorted(missing)}")

        # -----------------------------------------------------
        # Factor scores
        # -----------------------------------------------------
        if self.mode == "efa" and (not self.efa_oblique):
            # Orthogonal factors: eta ~ N(0, I)
            eta = numpyro.sample("eta", dist.Normal(0.0, 1.0).expand([n_obs, k]))
            L_omega = None
        else:
            # Correlated factors via LKJ on correlation
            L_omega = numpyro.sample("L_omega", dist.LKJCholesky(k, self.lkj_eta))
            numpyro.deterministic("factor_corr", L_omega @ L_omega.T)
            eta = numpyro.sample(
                "eta",
                dist.MultivariateNormal(loc=jnp.zeros(k), scale_tril=L_omega).expand([n_obs]),
            )  # [n_obs, k]

        # -----------------------------------------------------
        # Loadings
        # -----------------------------------------------------
        if self.mode == "efa":
            # Standard identified EFA:
            # - top-left k×k lower-triangular (strict-lower free; diag positive)
            # - remaining rows free on all k factors
            lam = jnp.zeros((n_items, k))

            if self._efa_n_free_lt > 0:
                lam_lt = numpyro.sample(
                    "lam_lt",
                    dist.Normal(0.0, self.loadings_scale).expand([self._efa_n_free_lt]),
                )
                lam = lam.at[self._efa_r_lt, self._efa_c_lt].set(lam_lt)

            lam_diag_raw = numpyro.sample("lam_diag_raw", dist.Normal(0.0, 0.5).expand([k]))
            lam = lam.at[jnp.arange(k), jnp.arange(k)].set(softplus(lam_diag_raw) + 0.1)

            if n_items > k:
                lam_rest = numpyro.sample(
                    "lam_rest",
                    dist.Normal(0.0, self.loadings_scale).expand([n_items - k, k]),
                )
                lam = lam.at[jnp.arange(k, n_items), :].set(lam_rest)

        else:
            # CFA:
            # - marker loading fixed to 1.0 (first item in each factor list)
            # - remaining primary loadings sampled (positive by default)
            lam = jnp.zeros((n_items, k))

            for f, idx_list in enumerate(self.item_to_factor_indices):
                idx_arr = jnp.asarray(idx_list, dtype=jnp.int32)
                if idx_arr.size == 0:
                    raise ValueError(f"CFA: factor {f} has no items assigned.")

                marker_item = int(idx_arr[0])
                lam = lam.at[marker_item, f].set(1.0)

                if idx_arr.size > 1:
                    nonmarker_idx = idx_arr[1:]

                    if self.allow_negative_cfa_loadings:
                        vals = numpyro.sample(
                            f"lam_f{f}_rest",
                            dist.Normal(0.0, self.loadings_scale).expand([nonmarker_idx.size]),
                        )
                    else:
                        vals = numpyro.sample(
                            f"lam_f{f}_rest",
                            dist.TruncatedNormal(
                                loc=self.cfa_loading_loc,
                                scale=self.loadings_scale,
                                low=0.0,
                            ).expand([nonmarker_idx.size]),
                        )

                    lam = lam.at[nonmarker_idx, f].set(vals)

        numpyro.deterministic("loadings", lam)

        # -----------------------------------------------------
        # Linear predictor (no item intercepts; thresholds carry item location)
        # -----------------------------------------------------
        mu = eta @ lam.T  # [n_obs, n_items]

        # -----------------------------------------------------
        # Thresholds: first finite threshold fixed to 0; monotone via softplus+cumsum
        # n_cats categories => n_cats-1 finite thresholds per item
        # -----------------------------------------------------
        if self.n_cats < 2:
            raise ValueError("n_cats must be >= 2.")

        if self.n_cats == 2:
            thresholds = jnp.zeros((n_items, 1))
        else:
            raw_delta = numpyro.sample(
                "raw_delta",
                dist.Normal(0.0, self.delta_sd).expand([n_items, self.n_cats - 2]),
            )
            delta = softplus(raw_delta) + self.threshold_offset
            thresholds = jnp.concatenate(
                [jnp.zeros((n_items, 1)), jnp.cumsum(delta, axis=1)],
                axis=1,
            )  # [n_items, n_cats-1]

        numpyro.deterministic("thresholds", thresholds)

        # -----------------------------------------------------
        # Likelihood (vectorized)
        # -----------------------------------------------------
        numpyro.sample("obs", OrderedProbit(mu, thresholds, validate_args=False), obs=y)

In [7]:
from numpyro.infer import MCMC, NUTS, init_to_feasible

# Dictionaries to hold results
mcmc_results_success = {}
idata_results_success = {}
mcmc_results_failure = {}
idata_results_failure = {}

# Data dict for looping
valence_data = {
    "success": preobs_success_array,
    "failure": preobs_failure_array,
}

for valence_name, data_array in valence_data.items():
    print(f"\nRunning models for {valence_name} valence...")

    Y = jnp.asarray(data_array, dtype=jnp.int32)
    n_items = Y.shape[1]
    # max_cat_per_item = jnp.max(Y, axis=0) + 1
    # n_cats = int(jnp.max(max_cat_per_item))
    n_cats = 6

    # Store results in appropriate dicts and set save directory
    if valence_name == "success":
        mcmc_results = mcmc_results_success
        idata_results = idata_results_success
    else:
        mcmc_results = mcmc_results_failure
        idata_results = idata_results_failure

    save_dir = Path(f"results/mcmc_idata/{valence_name}/new")
    save_dir.mkdir(parents=True, exist_ok=True)

    factor_range = range(2, 7)

    for n_factors in factor_range:
        print(f"Running {valence_name} model with {n_factors} factors...")

        rows_np, cols_np = np.tril_indices(n=n_items, k=-1, m=n_factors)
        n_free = rows_np.shape[0]
        
        init_values = {
            "lam_free":     jnp.zeros((n_free,)),
            "lam_diag_raw": jnp.zeros((n_factors,)),
            "eta":          jnp.zeros((Y.shape[0], n_factors)),
            "nu":           jnp.zeros((n_items,)),
        }
        if n_cats > 2:
            init_values["raw_delta"] = jnp.zeros((n_items, n_cats - 2))

        init_strategy = init_to_value(values=init_values)

        efa = BayesianOrdinalFactorModel(
            n_factors=n_factors,
            n_cats=n_cats,
            mode="efa",
        )

        nuts_kernel = NUTS(
            efa.model,
            init_strategy=init_strategy,
            target_accept_prob=0.9,
            max_tree_depth=10,
        )

        mcmc = MCMC(
            nuts_kernel,
            num_warmup=2000,
            num_samples=4000,
            num_chains=4,
            progress_bar=True,
        )

        # Y_min = int(jnp.min(Y))
        # Y_max = int(jnp.max(Y))
        
        # print("Y_min, Y_max:", Y_min, Y_max, "n_cats:", n_cats)
        
        # # If you have missing values coded as -1, this will be true:
        # assert Y_min >= 0, "Found negative category codes (likely missing values like -1)."
        
        # # Out-of-range categories
        # assert Y_max <= (n_cats - 1), "Found category codes >= n_cats."

        
        seed = 42 + (0 if valence_name == "success" else 1) * 10_000 + n_factors * 100
        rng_keys = jax.random.split(jax.random.PRNGKey(seed), mcmc.num_chains)
        mcmc.run(rng_keys, data=Y)

        mcmc_results[n_factors] = mcmc

        idata = az.from_numpyro(mcmc, log_likelihood=True)
        idata_results[n_factors] = idata

        # Save all stored idata to disk
        az.to_netcdf(idata, save_dir / f"{n_factors}.nc")

NameError: name 'preobs_success_array' is not defined

# Helper diagnostic functions

In [ ]:
# import numpy as np
# from scipy.stats import norm, multivariate_normal
# from scipy.optimize import minimize

# def polychoric_correlation(x, y, max_iter=10, tol=1e-4):
#     """
#     Compute polychoric correlation between two ordinal vectors x and y.
#     x, y: 1D numpy arrays of ordinal data (integers starting at 0)
#     Returns estimated correlation coefficient.
#     """
#     # Get contingency table
#     categories_x = np.unique(x)
#     categories_y = np.unique(y)
#     tab = np.zeros((len(categories_x), len(categories_y)))

#     for i, cat_x in enumerate(categories_x):
#         for j, cat_y in enumerate(categories_y):
#             tab[i, j] = np.sum((x == cat_x) & (y == cat_y))

#     # Compute thresholds for x
#     px = np.sum(tab, axis=1) / np.sum(tab)
#     thresh_x = np.concatenate(([-np.inf], norm.ppf(np.cumsum(px)[:-1]), [np.inf]))

#     # Compute thresholds for y
#     py = np.sum(tab, axis=0) / np.sum(tab)
#     thresh_y = np.concatenate(([-np.inf], norm.ppf(np.cumsum(py)[:-1]), [np.inf]))

#     # Negative log-likelihood function for correlation rho
#     def neg_log_lik(rho):
#         rho = float(np.squeeze(rho))
#         if not -0.999 < rho < 0.999:
#             return np.inf
#         ll = 0.0
#         for i in range(len(thresh_x)-1):
#             for j in range(len(thresh_y)-1):
#                 lower = [thresh_x[i], thresh_y[j]]
#                 upper = [thresh_x[i+1], thresh_y[j+1]]
#                 cdf_val = multivariate_normal.cdf(upper, mean=[0,0], cov=[[1,rho],[rho,1]]) - \
#                           multivariate_normal.cdf([lower[0], upper[1]], mean=[0,0], cov=[[1,rho],[rho,1]]) - \
#                           multivariate_normal.cdf([upper[0], lower[1]], mean=[0,0], cov=[[1,rho],[rho,1]]) + \
#                           multivariate_normal.cdf(lower, mean=[0,0], cov=[[1,rho],[rho,1]])
#                 # Avoid log(0)
#                 cdf_val = max(cdf_val, 1e-10)
#                 ll += tab[i,j] * np.log(cdf_val)
#         return -ll

#     res = minimize(neg_log_lik, 0.0, bounds=[(-0.999, 0.999)])
#     return res.x[0]


# def polychoric_corr_matrix(data):
#     """
#     Compute polychoric correlation matrix for ordinal data matrix
#     data: 2D numpy array (n_samples, n_items)
#     """
#     n_items = data.shape[1]
#     R = np.eye(n_items)
#     for i in range(n_items):
#         for j in range(i+1, n_items):
#             r = polychoric_correlation(data[:, i], data[:, j])
#             R[i, j] = r
#             R[j, i] = r
#     return R

# def plot_eigen_ppc(idata, n_factors=None, n_draws=200, output_folder=None,
#                    observed_data=None, compute_polychoric=False):
#     Sigma = reconstructed_latent_covariance(idata)

#     eig_rep = np.linalg.eigvalsh(Sigma)
#     n_draws = min(n_draws, eig_rep.shape[0])
#     idx = np.random.choice(eig_rep.shape[0], n_draws, replace=False)

#     plt.figure(figsize=(7, 4))
#     for i in idx:
#         plt.plot(eig_rep[i], color="gray", alpha=0.1)
#     plt.plot(np.mean(eig_rep, axis=0), color="black", lw=2, label="Posterior mean")

#     if compute_polychoric and observed_data is not None:
#         R = polychoric_corr_matrix(observed_data)
#         observed_eigvals = np.linalg.eigvalsh(R)
#         plt.plot(observed_eigvals, color="red", lw=2, linestyle='--', label="Observed eigenvalues (polychoric)")

#     plt.xlabel("Eigenvalue index")
#     plt.ylabel("Latent eigenvalue")
#     plt.title(f"Eigenvalue PPC — {n_factors}-factor model")
#     plt.legend()
#     plt.tight_layout()

#     if output_folder is not None:
#         eigenplot_path = output_folder / f"eigenplot_factors_{n_factors}.png"
#         plt.savefig(eigenplot_path, dpi=300)

#     plt.show()
#     plt.close()

# def diagnostics_by_param_family(idata, rhat_threshold=1.01, rhat_acceptable=1.05):
#     posterior = idata.posterior
#     param_names = list(posterior.data_vars)
    
#     records = []

#     for param in param_names:
#         data_array = posterior[param]

#         rhat = az.rhat(idata, var_names=[param], method="rank")[param].values
#         ess = az.ess(idata, var_names=[param], method="bulk")[param].values

#         rhat_flat = rhat.flatten()
#         rhat_flat = rhat_flat[~np.isnan(rhat_flat)]

#         ess_flat = ess.flatten()
#         ess_flat = ess_flat[~np.isnan(ess_flat)]

#         # Percentage of R-hat below threshold
#         rhat_pct_good = (
#             100 * np.mean(rhat_flat <= rhat_threshold)
#             if rhat_flat.size > 0 else np.nan
#         )

#         # Percentage of R-hat within acceptable range
#         rhat_pct_acceptable = (
#             100 * np.mean((rhat_threshold < rhat_flat) & (rhat_flat <= rhat_acceptable))
#             if rhat_flat.size > 0 else np.nan
#         )

#         # Percentage of R-hat problematic
#         rhat_pct_problematic = (
#             100 * np.mean(rhat_flat > rhat_acceptable)
#             if rhat_flat.size > 0 else np.nan
#         )

#         mean_ess = np.mean(ess_flat) if ess_flat.size > 0 else np.nan
#         median_ess = np.median(ess_flat) if ess_flat.size > 0 else np.nan
#         min_ess = np.min(ess_flat) if ess_flat.size > 0 else np.nan

#         records.append({
#             "Parameter": param,
#             f"% Rhat<{rhat_threshold:.2f}": rhat_pct_good,
#             f"% {rhat_threshold:.2f}<Rhat<{rhat_acceptable:.2f}": rhat_pct_acceptable,
#             f"% Rhat>{rhat_acceptable:.2f}": rhat_pct_problematic,
#             "Mean ESS": mean_ess,
#             "Median ESS": median_ess,
#             "Min ESS": min_ess,
#             "Num Params": rhat_flat.size
#         })

#     df = (
#         pd.DataFrame(records)
#         # .sort_values(f"% Rhat<{rhat_threshold:.2f}", ascending=False)
#     )

#     return df.round(2)

# def factor_variance(idata):
#     L = idata.posterior["loadings"].values          # (chains, draws, n_items, n_factors)
#     Phi = idata.posterior["factor_corr"].values     # (chains, draws, n_factors, n_factors)

#     L = L.reshape(-1, *L.shape[2:])
#     Phi = Phi.reshape(-1, *Phi.shape[2:])

#     var_f = np.einsum("dik,dkl,dil->dl", L, Phi, L)
#     return var_f.mean(axis=0)

# def reconstructed_latent_covariance(idata):
#     # Extract loadings L and factor correlation Phi
#     L = idata.posterior["loadings"].values  # (chains, draws, items, factors)
#     Phi = idata.posterior["factor_corr"].values  # (chains, draws, factors, factors)
    
#     # Extract psi_raw and transform it
#     psi_raw = idata.posterior["psi_raw"].values  # (chains, draws, items)
#     psi = jnn.softplus(psi_raw) + 0.1  # shape still (chains, draws, items)
    
#     # Reshape psi into a diagonal residual covariance matrix per draw
#     # First flatten chains and draws
#     n_draws = psi.shape[0] * psi.shape[1]
#     n_items = psi.shape[2]
#     psi = psi.reshape(n_draws, n_items)
#     L = L.reshape(n_draws, L.shape[2], L.shape[3])
#     Phi = Phi.reshape(n_draws, Phi.shape[2], Phi.shape[3])
    
#     cov_matrices = []
#     for i in range(n_draws):
#         # Residual covariance
#         Psi_matrix = np.diag(psi[i])  # diagonal matrix
        
#         # Factor covariance
#         factor_cov = L[i] @ Phi[i] @ L[i].T
        
#         # Total covariance = factor covariance + residual covariance
#         cov_total = factor_cov + Psi_matrix
#         cov_matrices.append(cov_total)
    
#     cov_matrices = np.array(cov_matrices)  # (n_draws, n_items, n_items)
#     return cov_matrices

# from factor_analyzer.rotator import Rotator

# def inspect_loadings(
#     idata,
#     rotation_type="none",
#     ci=0.90,
#     item_names=None,
#     factor_names=None,
#     flatten_columns=True,
#     show_sd=True,
#     show_ci=True,
#     summary_stat="mean",  # new argument: "mean" or "mode"
# ):
#     """
#     Inspect loadings from Bayesian ordinal EFA.

#     Parameters:
#     - idata: ArviZ InferenceData with posterior "loadings"
#              shape (chains, draws, items, factors)
#     - rotation_type: "none", "varimax", "promax", "oblimin", etc.
#     - ci: credible interval level (e.g. 0.90 or 90)
#     - item_names: list of item names
#     - factor_names: list of factor names
#     - flatten_columns: flatten MultiIndex columns
#     - show_sd: include posterior SD (ignored if flatten_columns=True)
#     - show_ci: include credible interval
#     - summary_stat: "mean" (default) or "mode" to summarize loadings

#     Returns:
#     - pandas DataFrame
#     """

#     import numpy as np
#     import pandas as pd
#     from factor_analyzer import Rotator
#     from scipy.stats import gaussian_kde

#     # Normalize CI if given in percentage
#     if ci > 1:
#         ci = ci / 100

#     L = idata.posterior["loadings"].values
#     samples = L.reshape(-1, L.shape[2], L.shape[3])  # (draws, items, factors)

#     if rotation_type != "none":
#         rotator = Rotator(method=rotation_type)
#         rotated_samples = np.array([rotator.fit_transform(sample) for sample in samples])
#     else:
#         rotated_samples = samples

#     def compute_mode(data_1d):
#         # Kernel Density Estimate based mode finder
#         kde = gaussian_kde(data_1d)
#         x_grid = np.linspace(np.min(data_1d), np.max(data_1d), 1000)
#         densities = kde(x_grid)
#         mode_index = np.argmax(densities)
#         return x_grid[mode_index]

#     if summary_stat == "mean":
#         summary_loadings = np.nanmean(rotated_samples, axis=0)
#     elif summary_stat == "mode":
#         n_items, n_factors = rotated_samples.shape[1], rotated_samples.shape[2]
#         summary_loadings = np.empty((n_items, n_factors))
#         for i in range(n_items):
#             for j in range(n_factors):
#                 summary_loadings[i, j] = compute_mode(rotated_samples[:, i, j])
#     else:
#         raise ValueError("summary_stat must be 'mean' or 'mode'")

#     sd_loadings = np.nanstd(rotated_samples, axis=0)

#     n_items, n_factors = summary_loadings.shape

#     if item_names is None:
#         item_names = [f"Item_{i+1}" for i in range(n_items)]
#     if factor_names is None:
#         factor_names = [f"F{i+1}" for i in range(n_factors)]

#     alpha = (1 - ci) / 2

#     data = []
#     for i in range(n_items):
#         row = []
#         for j in range(n_factors):
#             lower = np.nanquantile(rotated_samples[:, i, j], alpha)
#             upper = np.nanquantile(rotated_samples[:, i, j], 1 - alpha)

#             if flatten_columns:
#                 # Format with loading and CI in same string
#                 if show_ci:
#                     val = f"{summary_loadings[i, j]:.3f} ({lower:.3f}, {upper:.3f})"
#                 else:
#                     val = f"{summary_loadings[i, j]:.3f}"
#                 row.append(val)
#             else:
#                 # Use MultiIndex columns with separate Mean/Mode, SD, CI columns
#                 row.append(summary_loadings[i, j])
#                 if show_sd:
#                     row.append(sd_loadings[i, j])
#                 if show_ci:
#                     row.append(f"({lower:.3f}, {upper:.3f})")

#         data.append(row)

#     # Build columns
#     if flatten_columns:
#         cols = []
#         for f in factor_names:
#             if show_ci:
#                 cols.append(f"{f} {summary_stat.capitalize()} ({int(ci*100)}% CI)")
#             else:
#                 cols.append(f"{f} {summary_stat.capitalize()}")
#         df = pd.DataFrame(data, index=item_names, columns=cols)
#     else:
#         tuples = []
#         for factor in factor_names:
#             tuples.append((factor, summary_stat.capitalize()))
#             if show_sd:
#                 tuples.append((factor, "SD"))
#             if show_ci:
#                 tuples.append((factor, f"{int(ci*100)}% CI"))
#         columns = pd.MultiIndex.from_tuples(tuples)
#         df = pd.DataFrame(data, index=item_names, columns=columns)

#     return df


# def inspect_thresholds(
#     idata,
#     ci=0.90,
#     item_names=None,
#     summary_stat="mean",  # new argument: "mean" or "mode"
# ):
#     """
#     Inspect thresholds from Bayesian ordinal EFA.

#     Parameters:
#     - idata: ArviZ InferenceData with posterior "thresholds" (chains, draws, items, thresholds_per_item)
#     - ci: credible interval level (e.g. 0.90)
#     - item_names: list of item names (default: Item_1, Item_2, ...)
#     - summary_stat: "mean" (default) or "mode" to summarize thresholds

#     Returns:
#     - DataFrame with MultiIndex (Item, Threshold) and columns: Mean/Mode, CI lower, CI upper
#     """

#     import numpy as np
#     import pandas as pd
#     from scipy.stats import gaussian_kde

#     thresholds_samples = idata.posterior["thresholds"].values  # (chains, draws, items, thresholds)
#     n_chains, n_draws, n_items, n_thresholds = thresholds_samples.shape
#     samples_reshaped = thresholds_samples.reshape(n_chains * n_draws, n_items, n_thresholds)

#     def compute_mode(data_1d):
#         kde = gaussian_kde(data_1d)
#         x_grid = np.linspace(np.min(data_1d), np.max(data_1d), 1000)
#         densities = kde(x_grid)
#         mode_index = np.argmax(densities)
#         return x_grid[mode_index]

#     if summary_stat == "mean":
#         summary_vals = np.mean(samples_reshaped, axis=0)
#     elif summary_stat == "mode":
#         summary_vals = np.empty((n_items, n_thresholds))
#         for i in range(n_items):
#             for j in range(n_thresholds):
#                 summary_vals[i, j] = compute_mode(samples_reshaped[:, i, j])
#     else:
#         raise ValueError("summary_stat must be 'mean' or 'mode'")

#     lower_p = (1 - ci) / 2
#     upper_p = 1 - lower_p
#     lower_thresh = np.quantile(samples_reshaped, lower_p, axis=0)
#     upper_thresh = np.quantile(samples_reshaped, upper_p, axis=0)

#     if item_names is None:
#         item_names = [f"Item_{i+1}" for i in range(n_items)]

#     threshold_labels = [f"Threshold_{i+1}" for i in range(n_thresholds)]
#     index = pd.MultiIndex.from_product([item_names, threshold_labels], names=["Item", "Threshold"])

#     mean_flat = summary_vals.reshape(n_items * n_thresholds)
#     lower_flat = lower_thresh.reshape(n_items * n_thresholds)
#     upper_flat = upper_thresh.reshape(n_items * n_thresholds)

#     df = pd.DataFrame({
#         summary_stat.capitalize(): mean_flat,
#         f"{int(ci*100)}% CI Lower": lower_flat,
#         f"{int(ci*100)}% CI Upper": upper_flat,
#     }, index=index)

#     return df

# def communalities_posterior(idata=None, rotated_loadings=None, item_names=None, use_rotated=False):
#     """
#     Compute mean communalities and 95% credible intervals from posterior loadings
#     OR communalities directly from rotated loadings DataFrame.

#     Parameters:
#     - idata: ArviZ InferenceData object containing 'loadings' posterior samples.
#              Expected shape: (chains, draws, n_items, n_factors)
#     - rotated_loadings: pd.DataFrame of rotated loadings (items x factors) if use_rotated=True
#     - item_names: list of item names, length = n_items
#     - use_rotated: bool, if True, compute communalities from rotated_loadings DataFrame

#     Returns:
#     - DataFrame with columns: Item, Mean h², 2.5% h², 97.5% h², and SD h² (if posterior)
#     """
#     if use_rotated:
#         if rotated_loadings is None:
#             raise ValueError("rotated_loadings must be provided if use_rotated=True")
#         # rotated_loadings: DataFrame (items x factors)
#         communalities = (rotated_loadings ** 2).sum(axis=1)
#         df = pd.DataFrame({
#             "Item": item_names,
#             "Mean h²": communalities.values,
#             "2.5% h²": communalities.values,   # no CI here, just point estimate
#             "97.5% h²": communalities.values,
#             "SD h²": 0.0
#         })
#     else:
#         if idata is None:
#             raise ValueError("idata must be provided if use_rotated=False")

#         # Extract loadings samples (chains, draws, n_items, n_factors)
#         loadings_samples = idata.posterior["loadings"].values
#         n_chains, n_draws, n_items, n_factors = loadings_samples.shape
#         samples_reshaped = loadings_samples.reshape(n_chains * n_draws, n_items, n_factors)

#         communalities_samples = np.sum(samples_reshaped ** 2, axis=2)  # (samples, items)

#         mean_comm = np.mean(communalities_samples, axis=0)
#         sd_comm = np.std(communalities_samples, axis=0)
#         lower_comm = np.percentile(communalities_samples, 2.5, axis=0)
#         upper_comm = np.percentile(communalities_samples, 97.5, axis=0)

#         df = pd.DataFrame({
#             "Item": item_names,
#             "Mean h²": mean_comm,
#             "SD h²": sd_comm,
#             "2.5% h²": lower_comm,
#             "97.5% h²": upper_comm,
#         })

#     return df
    
# def inspect_factor_correlation(idata):
#     """
#     Extract and print the mean factor correlation matrix from an ArviZ InferenceData object.

#     Parameters:
#     - idata: ArviZ InferenceData object containing posterior samples.

#     Returns:
#     - mean_corr: numpy.ndarray of shape (n_factors, n_factors)
#     """
#     try:
#         corr_samples = idata.posterior["factor_corr"].values  # shape: (chains, draws, n_factors, n_factors)
#     except KeyError:
#         print("Warning: 'corr_mat' not found in idata.posterior. Returning identity matrix.")
#         # Infer number of factors from loadings shape or default to 2
#         try:
#             n_factors = idata.posterior["loadings"].shape[-1]
#         except Exception:
#             n_factors = 2
#         return np.eye(n_factors)

#     # Flatten chains and draws
#     corr_samples_reshaped = corr_samples.reshape(-1, corr_samples.shape[-2], corr_samples.shape[-1])
    
#     mean_corr = corr_samples_reshaped.mean(axis=0)

#     factor_labels = [f"F{i+1}" for i in range(mean_corr.shape[0])]
    
#     df_corr = pd.DataFrame(
#         mean_corr,
#         index=factor_labels,
#         columns=factor_labels
#     )
    
#     print("Mean factor correlation matrix:")
#     print(np.round(mean_corr, 2))
    
#     return df_corr

In [88]:
# import numpy as np
# import xarray as xr
# import warnings

# def _to_numpy_loadings(idata, var="loadings"):
#     da = idata.posterior[var]
#     other_dims = [d for d in da.dims if d not in ("chain", "draw")]
#     if len(other_dims) != 2:
#         raise ValueError(f"Expected {var} dims ('chain','draw',item,factor). Got {da.dims}")
#     item_dim, factor_dim = other_dims

#     L = da.transpose("chain", "draw", item_dim, factor_dim).values
#     n_chain, n_draw, n_items, k = L.shape
#     L = L.reshape(n_chain * n_draw, n_items, k)
#     return L, (n_chain, n_draw, n_items, k), (item_dim, factor_dim)

# def _sign_align_loadings(L, anchor="maxabs", anchor_items=None):
#     n_samples, n_items, k = L.shape

#     if anchor_items is None:
#         L_mean = L.mean(axis=0)
#         if anchor == "diag" and n_items >= k:
#             anchor_items = np.arange(k)
#         else:
#             anchor_items = np.argmax(np.abs(L_mean), axis=0)

#     anchor_items = np.asarray(anchor_items, dtype=int)
#     if anchor_items.shape[0] != k:
#         raise ValueError("anchor_items must have length k.")

#     anchor_vals = L[:, anchor_items, np.arange(k)]
#     signs = np.sign(anchor_vals)
#     signs = np.where(signs == 0.0, 1.0, signs)

#     return L * signs[:, None, :], anchor_items

# def _orthogonal_procrustes(L, L_ref, enforce_proper_rotation=True):
#     """
#     Per-sample orthogonal Procrustes:
#       minimize ||L R - L_ref||_F, subject to R'R = I
#     If enforce_proper_rotation=True, also require det(R)=+1 (no reflections).
#     """
#     n_samples, n_items, k = L.shape
#     L_rot = np.empty_like(L)
#     R_all = np.empty((n_samples, k, k), dtype=L.dtype)

#     for s in range(n_samples):
#         M = L[s].T @ L_ref
#         U, _, Vt = np.linalg.svd(M, full_matrices=False)
#         R = U @ Vt

#         if enforce_proper_rotation and np.linalg.det(R) < 0:
#             # Flip last singular vector to force det(R)=+1
#             U[:, -1] *= -1.0
#             R = U @ Vt

#         L_rot[s] = L[s] @ R
#         R_all[s] = R

#     return L_rot, R_all

# def tucker_phi_per_factor(A, B, eps=1e-12):
#     """
#     Tucker's congruence per factor between two loading matrices A and B.
#     A, B: (n_items, k)
#     Returns: (k,) congruence coefficients in [-1, 1]
#     """
#     A = np.asarray(A)
#     B = np.asarray(B)
#     if A.shape != B.shape:
#         raise ValueError(f"A and B must have same shape. Got {A.shape} vs {B.shape}")

#     num = np.sum(A * B, axis=0)
#     den = np.sqrt(np.sum(A * A, axis=0) * np.sum(B * B, axis=0)) + eps
#     return num / den

# def align_two_group_efa_optionA(
#     idata_success,
#     idata_failure,
#     loadings_var="loadings",
#     ref_group="success",
#     anchor="maxabs",
#     enforce_proper_rotation=True,
# ):
#     # Extract loadings
#     Ls, shape_s, dims_s = _to_numpy_loadings(idata_success, var=loadings_var)
#     Lf, shape_f, dims_f = _to_numpy_loadings(idata_failure, var=loadings_var)

#     if Ls.shape[1] != Lf.shape[1]:
#         raise ValueError(f"n_items mismatch: success={Ls.shape[1]} failure={Lf.shape[1]}")
#     if Ls.shape[2] != Lf.shape[2]:
#         raise ValueError(f"k mismatch: success={Ls.shape[2]} failure={Lf.shape[2]}")

#     n_items, k = Ls.shape[1], Ls.shape[2]

#     # Choose reference group samples
#     Lref_raw = Ls if ref_group == "success" else Lf

#     # Sign-align reference group to pick anchor items
#     Lref_signed, anchor_items = _sign_align_loadings(Lref_raw, anchor=anchor)
#     L_ref = Lref_signed.mean(axis=0)  # (n_items, k)

#     # Sign-align both groups using the same anchor items
#     Ls_signed, _ = _sign_align_loadings(Ls, anchor_items=anchor_items)
#     Lf_signed, _ = _sign_align_loadings(Lf, anchor_items=anchor_items)

#     # Procrustes align draws to shared reference
#     Ls_aligned, Rs = _orthogonal_procrustes(Ls_signed, L_ref, enforce_proper_rotation=enforce_proper_rotation)
#     Lf_aligned, Rf = _orthogonal_procrustes(Lf_signed, L_ref, enforce_proper_rotation=enforce_proper_rotation)

#     # Reshape back
#     n_chain_s, n_draw_s, _, _ = shape_s
#     n_chain_f, n_draw_f, _, _ = shape_f
#     Ls_aligned = Ls_aligned.reshape(n_chain_s, n_draw_s, n_items, k)
#     Lf_aligned = Lf_aligned.reshape(n_chain_f, n_draw_f, n_items, k)

#     item_dim_s, factor_dim_s = dims_s
#     item_dim_f, factor_dim_f = dims_f

#     da_s = xr.DataArray(
#         Ls_aligned,
#         dims=("chain", "draw", item_dim_s, factor_dim_s),
#         coords={
#             "chain": idata_success.posterior["chain"].values,
#             "draw": idata_success.posterior["draw"].values,
#             item_dim_s: idata_success.posterior[item_dim_s].values,
#             factor_dim_s: idata_success.posterior[factor_dim_s].values,
#         },
#         name=f"{loadings_var}_aligned",
#     )

#     da_f = xr.DataArray(
#         Lf_aligned,
#         dims=("chain", "draw", item_dim_f, factor_dim_f),
#         coords={
#             "chain": idata_failure.posterior["chain"].values,
#             "draw": idata_failure.posterior["draw"].values,
#             item_dim_f: idata_failure.posterior[item_dim_f].values,
#             factor_dim_f: idata_failure.posterior[factor_dim_f].values,
#         },
#         name=f"{loadings_var}_aligned",
#     )

#     # ----------------------------
#     # Tucker's phi per draw (paired draws; pairing is arbitrary across groups)
#     # ----------------------------   
#     Ls_draws = (
#         da_s.stack(sample=("chain", "draw"))
#             .transpose("sample", item_dim_s, factor_dim_s)
#             .values
#     )  # (S_s, n_items, k)
    
#     Lf_draws = (
#         da_f.stack(sample=("chain", "draw"))
#             .transpose("sample", item_dim_f, factor_dim_f)
#             .values
#     )  # (S_f, n_items, k)
    
#     S_s = Ls_draws.shape[0]
#     S_f = Lf_draws.shape[0]
#     S = min(S_s, S_f)
    
#     if S_s != S_f:
#         warnings.warn(
#             f"Unequal draw counts: success={S_s}, failure={S_f}. "
#             f"Truncating to {S} and pairing draws by index (arbitrary pairing)."
#         )
    
#     Ls_draws = Ls_draws[:S]
#     Lf_draws = Lf_draws[:S]
    
#     # Vectorized Tucker's phi: per draw, per factor
#     num = np.sum(Ls_draws * Lf_draws, axis=1)  # (S, k)
#     den = np.sqrt(
#         np.sum(Ls_draws * Ls_draws, axis=1) * np.sum(Lf_draws * Lf_draws, axis=1)
#     ) + 1e-12
#     phis = num / den  # (S, k)
    
#     phi_median = np.median(phis, axis=0)       # (k,)
#     phi_lo = np.quantile(phis, 0.05, axis=0)   # (k,)
#     phi_hi = np.quantile(phis, 0.95, axis=0)   # (k,)
#     p_gt_095 = np.mean(phis > 0.95, axis=0)    # (k,)
    
#     # ----------------------------
#     # Return dict (build explicitly)
#     # ----------------------------
#     return {
#         "L_ref":                    L_ref,
#         "anchor_items":             anchor_items,
#         "success_loadings_aligned": da_s,
#         "failure_loadings_aligned": da_f,
#         "success_rotations":        Rs,
#         "failure_rotations":        Rf,
#         "tucker_phi_draws":         phis,                 # (S, k)
#         "tucker_phi_median":        phi_median,           # (k,)
#         "tucker_phi_hdi90":         (phi_lo, phi_hi),     # ((k,), (k,))
#         "tucker_phi_p_gt_095":      p_gt_095,             # (k,)
#     }

In [90]:
# ============================================================
# Bayesian-congruent EFA post-processing workflow (NumPyro/ArviZ)
# - Select k via PSIS-LOO (predictive; Bayesian)
# - Align success vs failure posterior loadings (sign + proper Procrustes; Bayesian-safe post-processing)
# - Tucker's phi per posterior draw (Bayesian summary of congruence)
# - Summaries for loadings and thresholds (credible intervals)
# ============================================================

import warnings
import numpy as np
import pandas as pd
import xarray as xr
import arviz as az


# ----------------------------
# 1) k selection via PSIS-LOO
# ----------------------------
def loo_table(idata_by_k, scale="elpd_loo"):
    """
    idata_by_k: dict {k: InferenceData} with log_likelihood present.
    Returns a DataFrame sorted by best (highest) ELPD.
    """
    rows = []
    for k, idata in idata_by_k.items():
        loo = az.loo(idata, pointwise=False)
        # ArviZ returns elpd_loo and se, and p_loo, etc.
        rows.append({
            "k": k,
            "elpd_loo": float(loo.elpd_loo),
            "se_elpd_loo": float(loo.se),
            "p_loo": float(loo.p_loo),
        })
    df = pd.DataFrame(rows).sort_values("elpd_loo", ascending=False).reset_index(drop=True)
    # add deltas vs best
    best = df.loc[0, "elpd_loo"]
    df["delta_elpd"] = df["elpd_loo"] - best
    return df


# ----------------------------
# 2) Alignment + Tucker phi
# ----------------------------
def _to_samples_matrix(da, item_dim=None, factor_dim=None):
    """
    da: xarray DataArray with dims ('chain','draw', item_dim, factor_dim)
    Returns:
      X: np.ndarray (S, n_items, k) where S = chain*draw
      item_dim, factor_dim
    """
    if "chain" not in da.dims or "draw" not in da.dims:
        raise ValueError(f"Expected dims include ('chain','draw',...). Got {da.dims}")

    other = [d for d in da.dims if d not in ("chain", "draw")]
    if len(other) != 2:
        raise ValueError(f"Expected exactly two non-(chain,draw) dims. Got {other}")

    if item_dim is None or factor_dim is None:
        item_dim, factor_dim = other

    X = (
        da.transpose("chain", "draw", item_dim, factor_dim)
          .stack(sample=("chain", "draw"))
          .transpose("sample", item_dim, factor_dim)
          .values
    )
    return X, item_dim, factor_dim


def _sign_align_draws(L_draws, anchor_items):
    """
    L_draws: (S, n_items, k)
    anchor_items: (k,) item index per factor used to set sign.
    Returns sign-aligned L_draws.
    """
    S, n_items, k = L_draws.shape
    anchor_items = np.asarray(anchor_items, dtype=int)
    if anchor_items.shape[0] != k:
        raise ValueError("anchor_items must have length k.")

    anchor_vals = L_draws[:, anchor_items, np.arange(k)]
    signs = np.sign(anchor_vals)
    signs = np.where(signs == 0.0, 1.0, signs)
    return L_draws * signs[:, None, :]


def _choose_anchor_items(L_ref_draws, mode="maxabs"):
    """
    Choose anchor items per factor from reference group draws.
    mode:
      - 'maxabs': pick item with max |loading| in posterior mean
      - 'diag': item=f if n_items>=k else fallback to maxabs
    """
    L_mean = L_ref_draws.mean(axis=0)  # (n_items, k)
    n_items, k = L_mean.shape

    if mode == "diag" and n_items >= k:
        return np.arange(k)

    # default: max abs in mean per factor
    return np.argmax(np.abs(L_mean), axis=0)


def _orthogonal_procrustes_draws(L_draws, L_target, enforce_det_pos=True):
    """
    L_draws: (S, n_items, k)
    L_target: (n_items, k)
    Returns:
      L_rot: (S, n_items, k)
      R:     (S, k, k)
    """
    S, n_items, k = L_draws.shape
    L_rot = np.empty_like(L_draws)
    R_all = np.empty((S, k, k), dtype=L_draws.dtype)

    for s in range(S):
        M = L_draws[s].T @ L_target
        U, _, Vt = np.linalg.svd(M, full_matrices=False)
        R = U @ Vt

        if enforce_det_pos and np.linalg.det(R) < 0:
            U[:, -1] *= -1.0
            R = U @ Vt

        L_rot[s] = L_draws[s] @ R
        R_all[s] = R

    return L_rot, R_all


def tucker_phi_draws(Ls_draws, Lf_draws, warn_on_trunc=True):
    """
    Tucker's phi per draw and per factor.
    Ls_draws: (S_s, n_items, k)
    Lf_draws: (S_f, n_items, k)

    Since draws are independent across groups, pairing-by-index is arbitrary.
    We truncate to min length and warn (optional).
    Returns:
      phis: (S, k)
      summaries: dict with median, hdi90, p_gt_095
    """
    S_s = Ls_draws.shape[0]
    S_f = Lf_draws.shape[0]
    S = min(S_s, S_f)

    if warn_on_trunc and S_s != S_f:
        warnings.warn(
            f"Unequal draw counts: success={S_s}, failure={S_f}. "
            f"Truncating to {S} and pairing draws by index (arbitrary pairing)."
        )

    Ls = Ls_draws[:S]
    Lf = Lf_draws[:S]

    num = np.sum(Ls * Lf, axis=1)  # (S, k)
    den = np.sqrt(np.sum(Ls * Ls, axis=1) * np.sum(Lf * Lf, axis=1)) + 1e-12
    phis = num / den

    phi_median = np.median(phis, axis=0)
    phi_lo = np.quantile(phis, 0.05, axis=0)
    phi_hi = np.quantile(phis, 0.95, axis=0)
    p_gt_095 = np.mean(phis > 0.95, axis=0)

    return phis, {
        "phi_median": phi_median,
        "phi_hdi90": (phi_lo, phi_hi),
        "p_gt_095": p_gt_095,
    }


def align_success_failure_optionA(idata_success, idata_failure, loadings_var="loadings",
                                 ref_group="success", anchor_mode="maxabs",
                                 enforce_det_pos=True):
    """
    Bayesian-safe post-processing:
      - Choose anchor items from reference group mean loadings.
      - Sign-align draws within each group using those anchors.
      - Procrustes-align each draw to a shared reference matrix.
      - Compute Tucker's phi per draw (paired-by-index; arbitrary pairing).

    Returns a dict with aligned DataArrays and phi summaries.
    """
    da_s = idata_success.posterior[loadings_var]
    da_f = idata_failure.posterior[loadings_var]

    Ls_draws, item_dim_s, factor_dim_s = _to_samples_matrix(da_s)
    Lf_draws, item_dim_f, factor_dim_f = _to_samples_matrix(da_f)

    if Ls_draws.shape[1] != Lf_draws.shape[1]:
        raise ValueError(f"n_items mismatch: {Ls_draws.shape[1]} vs {Lf_draws.shape[1]}")
    if Ls_draws.shape[2] != Lf_draws.shape[2]:
        raise ValueError(f"k mismatch: {Ls_draws.shape[2]} vs {Lf_draws.shape[2]}")

    # pick reference group draws
    Lref_draws = Ls_draws if ref_group == "success" else Lf_draws

    anchor_items = _choose_anchor_items(Lref_draws, mode=anchor_mode)

    # sign-align both groups using same anchor items
    Ls_signed = _sign_align_draws(Ls_draws, anchor_items)
    Lf_signed = _sign_align_draws(Lf_draws, anchor_items)

    # shared reference matrix: mean of reference group after sign alignment
    L_ref = (Ls_signed if ref_group == "success" else Lf_signed).mean(axis=0)

    # procrustes align to shared L_ref
    Ls_aligned, Rs = _orthogonal_procrustes_draws(Ls_signed, L_ref, enforce_det_pos=enforce_det_pos)
    Lf_aligned, Rf = _orthogonal_procrustes_draws(Lf_signed, L_ref, enforce_det_pos=enforce_det_pos)

    # rebuild xarray DataArrays with original chain/draw dims
    n_chain_s = da_s.sizes["chain"]; n_draw_s = da_s.sizes["draw"]
    n_chain_f = da_f.sizes["chain"]; n_draw_f = da_f.sizes["draw"]
    n_items = Ls_aligned.shape[1]; k = Ls_aligned.shape[2]

    Ls_aligned = Ls_aligned.reshape(n_chain_s, n_draw_s, n_items, k)
    Lf_aligned = Lf_aligned.reshape(n_chain_f, n_draw_f, n_items, k)

    da_s_aligned = xr.DataArray(
        Ls_aligned,
        dims=("chain","draw", item_dim_s, factor_dim_s),
        coords={
            "chain": idata_success.posterior["chain"].values,
            "draw": idata_success.posterior["draw"].values,
            item_dim_s: da_s.coords[item_dim_s].values,
            factor_dim_s: da_s.coords[factor_dim_s].values,
        },
        name=f"{loadings_var}_aligned",
    )

    da_f_aligned = xr.DataArray(
        Lf_aligned,
        dims=("chain","draw", item_dim_f, factor_dim_f),
        coords={
            "chain": idata_failure.posterior["chain"].values,
            "draw": idata_failure.posterior["draw"].values,
            item_dim_f: da_f.coords[item_dim_f].values,
            factor_dim_f: da_f.coords[factor_dim_f].values,
        },
        name=f"{loadings_var}_aligned",
    )

    # Tucker phi per draw (paired-by-index; arbitrary pairing)
    LsA, _, _ = _to_samples_matrix(da_s_aligned, item_dim=item_dim_s, factor_dim=factor_dim_s)
    LfA, _, _ = _to_samples_matrix(da_f_aligned, item_dim=item_dim_f, factor_dim=factor_dim_f)
    phis, phi_summary = tucker_phi_draws(LsA, LfA, warn_on_trunc=True)

    return {
        "L_ref": L_ref,
        "anchor_items": anchor_items,
        "success_loadings_aligned": da_s_aligned,
        "failure_loadings_aligned": da_f_aligned,
        "success_rotations": Rs,
        "failure_rotations": Rf,
        "tucker_phi_draws": phis,          # (S,k)
        **phi_summary,                      # phi_median, phi_hdi90, p_gt_095
    }


# ----------------------------
# 3) Bayesian summaries for reporting
# ----------------------------
def summarize_loadings(da_loadings, ci=0.90, item_names=None, factor_names=None, stat="mean"):
    """
    da_loadings: xarray DataArray (chain, draw, item, factor) (aligned or raw)
    Returns a DataFrame with mean/median and credible intervals per loading.
    """
    if ci > 1:
        ci = ci / 100.0
    alpha = (1 - ci) / 2

    other = [d for d in da_loadings.dims if d not in ("chain","draw")]
    item_dim, factor_dim = other

    if stat == "mean":
        center = da_loadings.mean(("chain","draw"))
    elif stat == "median":
        center = da_loadings.median(("chain","draw"))
    else:
        raise ValueError("stat must be 'mean' or 'median'")

    lo = da_loadings.quantile(alpha, dim=("chain","draw"))
    hi = da_loadings.quantile(1-alpha, dim=("chain","draw"))

    n_items = da_loadings.sizes[item_dim]
    k = da_loadings.sizes[factor_dim]
    if item_names is None:
        item_names = [f"Item_{i+1}" for i in range(n_items)]
    if factor_names is None:
        factor_names = [f"F{j+1}" for j in range(k)]

    out = pd.DataFrame(index=item_names)
    for j, f in enumerate(factor_names):
        out[f"{f}_{stat}"] = center.isel({factor_dim: j}).values
        out[f"{f}_lo"] = lo.isel({factor_dim: j}).values
        out[f"{f}_hi"] = hi.isel({factor_dim: j}).values
    return out


def summarize_thresholds(idata, thresholds_var="thresholds", ci=0.90, item_names=None, stat="mean"):
    """
    idata.posterior[thresholds_var] expected dims: (chain, draw, item, threshold)
    Returns a long DataFrame: Item, Threshold, center, lo, hi
    """
    da = idata.posterior[thresholds_var]
    if ci > 1:
        ci = ci / 100.0
    alpha = (1 - ci) / 2

    other = [d for d in da.dims if d not in ("chain","draw")]
    if len(other) != 2:
        raise ValueError(f"Expected thresholds dims (chain,draw,item,threshold). Got {da.dims}")
    item_dim, thr_dim = other

    if stat == "mean":
        center = da.mean(("chain","draw"))
    elif stat == "median":
        center = da.median(("chain","draw"))
    else:
        raise ValueError("stat must be 'mean' or 'median'")

    lo = da.quantile(alpha, dim=("chain","draw"))
    hi = da.quantile(1-alpha, dim=("chain","draw"))

    n_items = da.sizes[item_dim]
    n_thr = da.sizes[thr_dim]
    if item_names is None:
        item_names = [f"Item_{i+1}" for i in range(n_items)]

    records = []
    for i in range(n_items):
        for t in range(n_thr):
            records.append({
                "Item": item_names[i],
                "Threshold": f"t{t+1}",
                stat: float(center.isel({item_dim: i, thr_dim: t}).values),
                "lo": float(lo.isel({item_dim: i, thr_dim: t}).values),
                "hi": float(hi.isel({item_dim: i, thr_dim: t}).values),
            })
    return pd.DataFrame.from_records(records)


def convergence_summary(idata, rhat_cut=1.01):
    """
    Basic Bayesian diagnostics for key parameters.
    Returns a small DataFrame: var, max_rhat, min_ess_bulk.
    """
    rh = az.rhat(idata, method="rank")
    ess = az.ess(idata, method="bulk")

    rows = []
    for v in idata.posterior.data_vars:
        r = rh[v].values
        e = ess[v].values
        r = r[np.isfinite(r)]
        e = e[np.isfinite(e)]
        rows.append({
            "var": v,
            "max_rhat": float(np.max(r)) if r.size else np.nan,
            "min_ess_bulk": float(np.min(e)) if e.size else np.nan,
            "rhat_ok": bool(np.max(r) <= rhat_cut) if r.size else True,
        })
    return pd.DataFrame(rows).sort_values(["rhat_ok","max_rhat"], ascending=[False, True])


# ============================================================
# Example workflow
# ============================================================
# 1) After fitting models for k=2..6:
#   idata_success_by_k = {2: idata2, 3: idata3, ...}
#   idata_failure_by_k = {2: idata2, 3: idata3, ...}
#
# print(loo_table(idata_success_by_k))
# print(loo_table(idata_failure_by_k))
#
# 2) Choose k* and take the corresponding idata objects:
#   id_s = idata_success_by_k[k_star]
#   id_f = idata_failure_by_k[k_star]
#
# 3) Align + phi:
# out = align_success_failure_optionA(id_s, id_f, loadings_var="loadings",
#                                    ref_group="success", anchor_mode="maxabs",
#                                    enforce_det_pos=True)
#
# 4) Summaries:
# load_s = summarize_loadings(out["success_loadings_aligned"], ci=0.90, stat="mean")
# load_f = summarize_loadings(out["failure_loadings_aligned"], ci=0.90, stat="mean")
# thr_s = summarize_thresholds(id_s, ci=0.90, stat="mean")
# thr_f = summarize_thresholds(id_f, ci=0.90, stat="mean")
#
# print(out["phi_median"], out["phi_hdi90"], out["p_gt_095"])
# print(convergence_summary(id_s).head(10))

# Success Attributions EFA

In [ ]:
output_folder_success = Path("outputs/efa/success/new")

os.makedirs(output_folder_success, exist_ok=True)

In [ ]:
load_dir = Path("results/mcmc_idata/success/new")

idata_results_success = {
    path.stem: az.from_netcdf(path)
    for path in load_dir.glob("*.nc")
}

In [ ]:
# Print inference summary and traceplots
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # Ignore all warnings
    for n_factors, idata in idata_results_success.items():
        print(f"Summary: {n_factors} factors:")
        summary_str = az.summary(idata).to_string()
        print(summary_str)  # prints to console
        
        # Save to file
        summary_path = output_folder_success/ f"summary_factors_{n_factors}.txt"
        with open(summary_path, "w") as f:
            f.write(summary_str)

        az.plot_trace(idata)
        plt.tight_layout()

        # Save figure
        traceplot_path = output_folder_success / f"traceplot_factors_{n_factors}.png"
        plt.savefig(traceplot_path)      
        plt.show()
        plt.close()

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # Ignore all warnings
    for n_factors, idata in idata_results_success.items():
        diagnostics_df = diagnostics_by_param_family(idata, rhat_threshold=1.01, rhat_acceptable=1.05)
        print(f"Factors {n_factors}:")
        print(diagnostics_df)

        filename = output_folder_success / f"diagnostics_factors_{n_factors}.tex"
    
        latex_body = diagnostics_df.to_latex(index=False, escape=True, float_format="{:.2f}".format)

        sideways = f"""
        \\begin{{sidewaystable}}[htbp]
        \\centering
        \\caption{{Diagnostics summary for {n_factors}-factor model (Success)}}
        {latex_body}
        \\end{{sidewaystable}}
        """
        
        with open(filename, "w") as f:
            f.write(sideways)
        
        print(f"Saved diagnostics table for {n_factors} factors to {filename}")
        print("\n")

In [ ]:
fit_stats_success = []

with warnings.catch_warnings():
    warnings.simplefilter("ignore")

    for n_factors, idata in idata_results_success.items():
        loo = az.loo(idata)
        waic = az.waic(idata)

        fit_stats_success.append({
            "n_factors": n_factors,

            # LOO
            "elpd_loo": loo.elpd_loo,
            "se_elpd_loo": loo.se,
            "p_loo": loo.p_loo,

            # WAIC
            "elpd_waic": waic.elpd_waic,
            "se_elpd_waic": waic.se,
            "p_waic": waic.p_waic,
        })

loo_waic_df_success = (
    pd.DataFrame(fit_stats_success)
    .sort_values("elpd_loo", ascending=False)
    .reset_index(drop=True)
)

print(loo_waic_df_success)

# ---- Save LaTeX table ----
filename_loo_waic_success = output_folder_success / "LOO_WAIC.tex"

latex_code_loo_waic_success = loo_waic_df_success.to_latex(
    index=False,
    float_format="%.2f",
    escape=True
)

with open(filename_loo_waic_success, "w") as f:
    f.write(latex_code_loo_waic_success)

print(f"Saved LOO and WAIC to {filename_loo_waic_success}")

In [ ]:
# Plot ELPD LOO with error bars
plt.figure(figsize=(6,4))
plt.errorbar(
    loo_waic_df_success["n_factors"], 
    loo_waic_df_success["elpd_loo"], 
    yerr=loo_waic_df_success["se_elpd_loo"], 
    fmt='o-', capsize=5
)
plt.xlabel("Number of Factors")
plt.ylabel("ELPD LOO (higher is better)")
plt.title("LOO Model Comparison with SE")
plt.grid(True)
plt.xticks(loo_waic_df_success["n_factors"])  # ensure all factor counts shown

plt.tight_layout()
LOOplot_path_success = output_folder_success / "LOOplot.png"
plt.savefig(LOOplot_path_success)
plt.show()
plt.close()

# Compute and print LOO improvement differences between consecutive models (descending order)
loo_diffs_success = np.diff(loo_waic_df_success["elpd_loo"])
print("\nLOO improvement differences between consecutive models (descending order):")
for i, diff in enumerate(loo_diffs_success, 1):
    print(f"{int(loo_waic_df_success.loc[i-1, 'n_factors'])} → {int(loo_waic_df_success.loc[i, 'n_factors'])}: {diff:.3f}")

# Save the summary table
table_path_success = output_folder_success / "model_fit_stats_success.csv"
loo_waic_df_success.to_csv(table_path_success, index=False)
print(f"\nSummary statistics saved to: {table_path_success}")

In [ ]:
# Build a table summarizing factor variances across models
factor_var_records = []

for n_factors, idata in idata_results.items():
    var_f = factor_variance(idata)
    factor_var_dict = {f"FactorVar_{i+1}": v for i, v in enumerate(var_f)}
    factor_var_dict["n_factors"] = n_factors
    factor_var_records.append(factor_var_dict)

factor_vars_df = pd.DataFrame(factor_var_records)
factor_vars_df = factor_vars_df.sort_values(by="n_factors").reset_index(drop=True)

# Convert to long format for easy plotting or grouping
factor_vars_df_long = factor_vars_df.melt(
    id_vars="n_factors",
    var_name="factor_num",
    value_name="variance"
)

# Clean factor_num column (e.g. "FactorVar_1" → 1)
factor_vars_df_long["factor_num"] = factor_vars_df_long["factor_num"].str.replace("FactorVar_", "").astype(int)

# Sort the long dataframe nicely
factor_vars_df_long = factor_vars_df_long.sort_values(["n_factors", "factor_num"]).reset_index(drop=True)

# Calculate cumulative variance per model (factor solution)
factor_vars_df_long["cumulative_variance"] = factor_vars_df_long.groupby("n_factors")["variance"].cumsum()

# Print out variance info per model
for n_factors, group_df in factor_vars_df_long.groupby("n_factors"):
    print(f"\nFactor variance details for model with {n_factors} factors:\n")
    display_df = group_df.drop(columns="n_factors").reset_index(drop=True)
    print(display_df)

In [ ]:
for k, idata in idata_results_success.items():
    plot_eigen_ppc(idata_results_success[k], n_factors=5, n_draws=200, observed_data=preobs_success_array, compute_polychoric=True)

In [ ]:
rotated_loadings_dfs_success = {}

for k, idata in idata_results_success.items():
    print(f"\n{k} factors — loadings: Oblimin Rotated")
    df_loadings = inspect_loadings(idata, rotation_type="oblimin", item_names=item_cols, show_sd=False, ci=0.90, show_ci=True, summary_stat="mode")
    print(df_loadings)

    rotated_loadings_dfs_success[k] = df_loadings

    # ---- SAVE AS LATEX ----
    tex_path = output_folder_success / f"loadings_success_{k}f.tex"

    latex_body = df_loadings.to_latex(index=False, escape=True, float_format="{:.2f}".format)

    sideways = f"""
        \\begin{{sidewaystable}}[htbp]
        \\centering
        \\caption{{Oblimin-rotated factor loadings for the {n_factors}-factor EFA model (success valence)}}
        {latex_body}
        \\end{{sidewaystable}}
        """

    print(f"Saved LaTeX table to: {tex_path}")

    # print(f"\n{k} factors — thresholds:")
    # df_thresholds = inspect_thresholds(idata, item_names=item_cols)
    # print(df_thresholds)

In [ ]:
rotated_loadings_dfs_success = {}

for k, idata in idata_results_success.items():
    print(f"\n{k} factors — loadings: Geomin Rotated")
    df_loadings = inspect_loadings(idata, rotation_type="geomin_obl", item_names=item_cols, show_sd=False, show_ci=False)
    print(df_loadings)

    rotated_loadings_dfs_success[k] = df_loadings

    # print(f"\n{k} factors — thresholds:")
    # df_thresholds = inspect_thresholds(idata, item_names=item_cols)
    # print(df_thresholds)

In [ ]:
# Before rotation 
for k, idata in idata_results_success.items():
    df = communalities_posterior(idata=idata, item_names=item_cols)  # Now handles length mismatches gracefully
    
    print(f"\n{k} factors – communalities:")
    print(df)
    
    # Save as LaTeX table
    latex_table = df.to_latex(index=False, escape=False, column_format="l c c", float_format="%.3f")
    latex_path = output_folder / f"communalities_{k}_factors.tex"
    with open(latex_path, "w") as f:
        f.write(latex_table)
    print(f"Saved LaTeX table to: {latex_path}")

In [ ]:
# After rotation 
for k, loadings_df in rotated_loadings_dfs_success.items():
    df = communalities_posterior(rotated_loadings=loadings_df, item_names=item_cols, use_rotated=True)  # Now handles length mismatches gracefully
    
    print(f"\n{k} factors – post rotation communalities:")
    print(df)
    
    # Save as LaTeX table
    latex_table = df.to_latex(index=False, escape=False, column_format="l c c", float_format="%.3f")
    latex_path = output_folder / f"communalities_{k}_factors.tex"
    with open(latex_path, "w") as f:
        f.write(latex_table)
    print(f"Saved LaTeX table to: {latex_path}")

In [ ]:
# Example usage for all idata_results
for k, idata in idata_results_success.items():
    print(f"\nFactor correlation matrix for model with {k} factors:")
    df_corr = inspect_factor_correlation(idata)

    print(df_corr)
    
    # ---- SAVE AS LATEX ----
    tex_path = output_folder_success / f"factor_correlations_success_{k}f.tex"

    df_corr.to_latex(
        tex_path,
        index=True,
        float_format="%.3f",
        caption=f"Posterior mean factor correlation matrix for the {k}-factor EFA model (success valence).",
        label=f"tab:factor_corr_success_{k}f",
        column_format="l" + "r" * df_corr.shape[1],
        escape=False
    )

    print(f"Saved LaTeX table to: {tex_path}")    

# Failure Attributions EFA

In [ ]:
output_folder_failure = Path("outputs/efa/failure/new")

os.makedirs(output_folder_failure, exist_ok=True)

In [ ]:
load_dir = Path("results/mcmc_idata/failure/new")

idata_results_failure = {
    path.stem: az.from_netcdf(path)
    for path in load_dir.glob("*.nc")
}

In [ ]:
# Print inference summary and traceplots
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # Ignore all warnings
    for n_factors, idata in idata_results_failure.items():
        print(f"Summary: {n_factors} factors:")
        summary_str = az.summary(idata).to_string()
        print(summary_str)  # prints to console
        
        # Save to file
        summary_path = output_folder_failure / f"summary_factors_{n_factors}.txt"
        with open(summary_path, "w") as f:
            f.write(summary_str)

        az.plot_trace(idata)
        plt.tight_layout()

        # Save figure
        traceplot_path = output_folder_failure / f"traceplot_factors_{n_factors}.png"
        plt.savefig(traceplot_path)      
        plt.show()
        plt.close()

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # Ignore all warnings
    for n_factors, idata in idata_results_failure.items():
        diagnostics_df = diagnostics_by_param_family(idata, rhat_threshold=1.01, rhat_acceptable=1.05)
        print(f"Factors {n_factors}:")
        print(diagnostics_df)

        filename = output_folder_failure / f"diagnostics_factors_{n_factors}.tex"

        latex_body = diagnostics_df.to_latex(index=False, escape=True, float_format="{:.2f}".format)

        sideways = f"""
        \\begin{{sidewaystable}}[htbp]
        \\centering
        \\caption{{Diagnostics summary for {n_factors}-factor model (Failure)}}
        {latex_body}
        \\end{{sidewaystable}}
        """
        
        with open(filename, "w") as f:
            f.write(sideways)
        
        print(f"Saved diagnostics table for {n_factors} factors to {filename}")
        print("\n")

In [ ]:
fit_stats_failure = []

with warnings.catch_warnings():
    warnings.simplefilter("ignore")

    for n_factors, idata in idata_results_failure.items():
        loo = az.loo(idata)
        waic = az.waic(idata)

        fit_stats_failure.append({
            "n_factors": n_factors,

            # LOO
            "elpd_loo": loo.elpd_loo,
            "se_elpd_loo": loo.se,
            "p_loo": loo.p_loo,

            # WAIC
            "elpd_waic": waic.elpd_waic,
            "se_elpd_waic": waic.se,
            "p_waic": waic.p_waic,
        })

loo_waic_df_failure = (
    pd.DataFrame(fit_stats_failure)
    .sort_values("elpd_loo", ascending=False)
    .reset_index(drop=True)
)

print(loo_waic_df_failure)

# ---- Save LaTeX table ----
filename_loo_waic_failure = output_folder_failure / "LOO_WAIC.tex"

latex_code_loo_waic_failure = loo_waic_df_failure.to_latex(
    index=False,
    float_format="%.2f",
    escape=True
)

with open(filename_loo_waic_failure, "w") as f:
    f.write(latex_code_loo_waic_failure)

print(f"Saved LOO and WAIC to {filename_loo_waic_failure}")

In [ ]:
# Plot ELPD LOO with error bars
plt.figure(figsize=(6,4))
plt.errorbar(
    loo_waic_df_failure["n_factors"], 
    loo_waic_df_failure["elpd_loo"], 
    yerr=loo_waic_df_failure["se_elpd_loo"], 
    fmt='o-', capsize=5
)
plt.xlabel("Number of Factors")
plt.ylabel("ELPD LOO (higher is better)")
plt.title("LOO Model Comparison with SE")
plt.grid(True)
plt.xticks(loo_waic_df_failure["n_factors"])  # ensure all factor counts shown

plt.tight_layout()
LOOplot_path_failure = output_folder_failure / "LOOplot.png"
plt.savefig(LOOplot_path_failure)
plt.show()
plt.close()

# Compute and print LOO improvement differences between consecutive models (descending order)
loo_diffs_failure = np.diff(loo_waic_df_failure["elpd_loo"])
print("\nLOO improvement differences between consecutive models (descending order):")
for i, diff in enumerate(loo_diffs_failure, 1):
    print(f"{int(loo_waic_df_failure.loc[i-1, 'n_factors'])} → {int(loo_waic_df_failure.loc[i, 'n_factors'])}: {diff:.3f}")

# Save the summary table
table_path_failure = output_folder_failure / "model_fit_stats_failure.csv"
loo_waic_df_failure.to_csv(table_path_failure, index=False)
print(f"\nSummary statistics saved to: {table_path_failure}")

In [ ]:
# Build a table summarizing factor variances across models
factor_var_records = []

for n_factors, idata in idata_results.items():
    var_f = factor_variance(idata)
    factor_var_dict = {f"FactorVar_{i+1}": v for i, v in enumerate(var_f)}
    factor_var_dict["n_factors"] = n_factors
    factor_var_records.append(factor_var_dict)

factor_vars_df = pd.DataFrame(factor_var_records)
factor_vars_df = factor_vars_df.sort_values(by="n_factors").reset_index(drop=True)

# Convert to long format for easy plotting or grouping
factor_vars_df_long = factor_vars_df.melt(
    id_vars="n_factors",
    var_name="factor_num",
    value_name="variance"
)

# Clean factor_num column (e.g. "FactorVar_1" → 1)
factor_vars_df_long["factor_num"] = factor_vars_df_long["factor_num"].str.replace("FactorVar_", "").astype(int)

# Sort the long dataframe nicely
factor_vars_df_long = factor_vars_df_long.sort_values(["n_factors", "factor_num"]).reset_index(drop=True)

# Calculate cumulative variance per model (factor solution)
factor_vars_df_long["cumulative_variance"] = factor_vars_df_long.groupby("n_factors")["variance"].cumsum()

# Print out variance info per model
for n_factors, group_df in factor_vars_df_long.groupby("n_factors"):
    print(f"\nFactor variance details for model with {n_factors} factors:\n")
    display_df = group_df.drop(columns="n_factors").reset_index(drop=True)
    print(display_df)

In [ ]:
for k, idata in idata_results_success.items():
    plot_eigen_ppc(idata, n_factors=k)

In [ ]:
rotated_loadings_dfs_failure = {}

for k, idata in idata_results_failure.items():
    print(f"\n{k} factors — loadings: Oblimin Rotated")
    df_loadings = inspect_loadings(idata, rotation_type="oblimin", item_names=item_cols, 
                                   ci=0.90, show_sd=False, show_ci=True, flatten_columns=True)
    print(df_loadings)

    rotated_loadings_dfs_failure[k] = df_loadings

    # ---- SAVE AS LATEX ----
    tex_path = output_folder_failure / f"loadings_failure_{k}f.tex"

    latex_body = df_loadings.to_latex(index=False, escape=True, float_format="{:.2f}".format)

    sideways = f"""
    \\begin{{sidewaystable}}[htbp]
    \\centering
    \\caption{{Oblimin-rotated factor loadings for the {n_factors}-factor EFA model (failure valence)}}
    {latex_body}
    \\end{{sidewaystable}}
    """

    print(f"Saved LaTeX table to: {tex_path}")

    # print(f"\n{k} factors — thresholds:")
    # df_thresholds = inspect_thresholds(idata, item_names=item_cols)
    # print(df_thresholds)

In [ ]:
rotated_loadings_dfs_failure = {}

for k, idata in idata_results_failure.items():
    print(f"\n{k} factors — loadings: Geomin Rotated")
    df_loadings = inspect_loadings(idata, rotation_type="geomin_obl", item_names=item_cols, show_sd=False, show_ci=False)
    print(df_loadings)

    rotated_loadings_dfs_failure[k] = df_loadings

    # print(f"\n{k} factors — thresholds:")
    # df_thresholds = inspect_thresholds(idata, item_names=item_cols)
    # print(df_thresholds)

In [ ]:
# Before rotation 
for k, idata in idata_results_failure.items():
    df = communalities_posterior(idata=idata, item_names=item_cols)  # Now handles length mismatches gracefully
    
    print(f"\n{k} factors – communalities:")
    print(df)
    
    # Save as LaTeX table
    latex_table = df.to_latex(index=False, escape=False, column_format="l c c", float_format="%.3f")
    latex_path = output_folder_failure / f"communalities_{k}_factors_failure_pre.tex"
    with open(latex_path, "w") as f:
        f.write(latex_table)
    print(f"Saved LaTeX table to: {latex_path}")

In [ ]:
# After rotation 
for k, loadings_df in rotated_loadings_dfs_failure.items():
    df = communalities_posterior(rotated_loadings=loadings_df, item_names=item_cols, use_rotated=True)  # Now handles length mismatches gracefully
    
    print(f"\n{k} factors – post rotation communalities:")
    print(df)
    
    # Save as LaTeX table
    latex_table = df.to_latex(index=False, escape=False, column_format="l c c", float_format="%.3f")
    latex_path = output_folder_failure / f"communalities_{k}_factors_failure_post.tex"
    with open(latex_path, "w") as f:
        f.write(latex_table)
    print(f"Saved LaTeX table to: {latex_path}")

In [ ]:
# Example usage for all idata_results
for k, idata in idata_results_failure.items():
    print(f"\nFactor correlation matrix for model with {k} factors:")
    df_corr = inspect_factor_correlation(idata)

    print(df_corr)
    
    # ---- SAVE AS LATEX ----
    tex_path = output_folder_failure / f"factor_correlations_failure_{k}f.tex"

    df_corr.to_latex(
        tex_path,
        index=True,
        float_format="%.2f",
        caption=f"Posterior mean factor correlation matrix for the {k}-factor EFA model (success valence).",
        label=f"tab:factor_corr_success_{k}f",
        column_format="l" + "r" * df_corr.shape[1],
        escape=False
    )

    print(f"Saved LaTeX table to: {tex_path}")    

# CFA

In [91]:
# class BayesianOrdinalMGCFA:
#     def __init__(
#         self,
#         n_factors: int,
#         n_cats: int,
#         group: jnp.ndarray,
#         n_groups: int,
#         model_type: str = "cfa",          # "cfa" or "null"
#         item_to_factor_indices: Optional[Sequence[jnp.ndarray]] = None,        
#         invariance: Optional[str] = "metric",  # "configural", "metric", "scalar"
#         cross_loadings: bool = False,
#         loading_loc_prior: float = 0.6,
#         loading_scale_prior: float = 0.2,
#         cross_loading_scale: float = 0.05,
#         min_threshold_spacing: float = 0.05,
#         df_latent: float = 7.0,
#         partial_invariance: Optional[dict] = None,
#     ):
#         self.n_factors = n_factors
#         self.n_cats = n_cats
#         self.item_to_factor_indices = item_to_factor_indices
#         self.group = group
#         self.n_groups = n_groups
#         self.model_type = model_type.lower()
#         self.invariance = invariance.lower() if invariance is not None else None
#         self.cross_loadings = cross_loadings
#         self.loading_loc_prior = loading_loc_prior
#         self.loading_scale_prior = loading_scale_prior
#         self.cross_loading_scale = cross_loading_scale
#         self.min_threshold_spacing = min_threshold_spacing
#         self.df_latent = df_latent
#         self.partial_invariance = partial_invariance or {
#             "loadings_free_factors": [],
#             "intercepts_free_factors": [],
#             "variances_free_factors": [],
#             "residuals_free_factors": [],  # Placeholder, not implemented
#         }

#         # NULL model shortcut
#         if self.model_type == "null":
#             self.primary_mask = None
#             return
        
#         assert self.model_type in ("null", "cfa")
#         assert self.invariance in ("configural", "metric", "scalar")

#         n_items = sum(len(idx) for idx in item_to_factor_indices)
#         mask = jnp.zeros((n_items, n_factors), dtype=bool)
#         for f, idx in enumerate(item_to_factor_indices):
#             mask = mask.at[idx, f].set(True)
#         self.primary_mask = mask
#         self.n_items = n_items

#     def model(self, data: jnp.ndarray):
#         n_obs, n_items = data.shape
#         y = data
#         group = self.group
#         G = self.n_groups
#         F = self.n_factors

#         # --------------------------
#         # Thresholds (Intercepts)
#         # --------------------------

#         # Sample tau0: shape depends on invariance
#         tau0 = numpyro.sample(
#             "tau0",
#             dist.Normal(0.0, 1.0).expand(
#                 [G, n_items] if self.invariance != "scalar" else [n_items]
#             ),
#         )
#         if self.invariance == "scalar":
#             tau0 = jnp.broadcast_to(tau0, (G, n_items))

#         # For intercept partial invariance: determine which items belong to free factors
#         intercepts_free_factors = self.partial_invariance.get("intercepts_free_factors", [])
#         free_intercept_items = jnp.zeros(n_items, dtype=bool)
#         for f in intercepts_free_factors:
#             free_intercept_items = free_intercept_items.at[self.item_to_factor_indices[f]].set(True)

#         # raw_delta always shape [G, n_items, n_cats-2]
#         raw_delta = numpyro.sample(
#             "raw_delta",
#             dist.Normal(0.0, 1.0).expand([G, n_items, self.n_cats - 2]),
#         )
#         delta = softplus(raw_delta) + self.min_threshold_spacing

#         # Construct thresholds by cumulative sum
#         thresholds = jnp.cumsum(
#             jnp.concatenate([jnp.zeros((G, n_items, 1)), delta], axis=-1),
#             axis=-1,
#         )
#         # Add intercept offsets tau0
#         thresholds = thresholds + tau0[..., None]

#         # Fix first threshold for identification
#         thresholds = thresholds.at[..., 0].set(0.0)

#         # Partial invariance on intercepts:
#         # For factors *not* free, set thresholds group 1..G equal to group 0 for their items
#         for f in range(F):
#             if f not in intercepts_free_factors:
#                 idxs = self.item_to_factor_indices[f]
#                 thresholds = thresholds.at[1:, idxs, :].set(thresholds[0, idxs, :])

#         # Ensure thresholds are ordered (monotonic)
#         thresholds = jnp.sort(thresholds, axis=-1)

#         # --------------------------
#         # Null model (no factors)
#         # --------------------------

#         if self.model_type == "null":
#             mu = jnp.zeros((n_obs, n_items))
#             lik = OrderedProbit(mu, thresholds[group])
#             numpyro.sample("obs", lik, obs=y)
#             numpyro.deterministic("log_likelihood", lik.log_prob(y).sum(axis=1))
#             return

#         # --------------------------
#         # Latent means (group differences)
#         # --------------------------

#         latent_means = jnp.zeros((G, F))
#         z_latent_means = numpyro.sample(
#             "z_latent_means",
#             dist.StudentT(self.df_latent, 0.0, 1.0).expand([G - 1, F]),
#         )
#         latent_means = latent_means.at[1:].set(z_latent_means)

#         # --------------------------
#         # Factor covariance
#         # --------------------------

#         L_corr = numpyro.sample(
#             "L_corr",
#             dist.LKJCholesky(dimension=F, concentration=2.0),
#         )

#         sigma_g = numpyro.sample(
#             "sigma_g",
#             dist.HalfNormal(0.5).expand([G, F]),
#         )

#         # Apply partial invariance on variances
#         variances_free_factors = self.partial_invariance.get("variances_free_factors", [])
#         for f in range(F):
#             if f not in variances_free_factors:
#                 # Constrain variances equal across groups (groups 1..G equal to group 0)
#                 sigma_g = sigma_g.at[1:, f].set(sigma_g[0, f])

#         L_omega_g = L_corr * sigma_g[:, None, :]
#         cov_g = jnp.einsum("gij,gkj->gik", L_omega_g, L_omega_g)

#         # --------------------------
#         # Factor loadings
#         # --------------------------

#         lam_list = []
#         for g in range(G):
#             lam_g = jnp.zeros((n_items, F))
#             for f, idx in enumerate(self.item_to_factor_indices):
#                 vals = numpyro.sample(
#                     f"lam_g{g}_f{f}",
#                     dist.TruncatedNormal(
#                         loc=self.loading_loc_prior,
#                         scale=self.loading_scale_prior,
#                         low=0.0,
#                     ).expand([len(idx)]),
#                 )
#                 # If metric or scalar invariance but this factor is not free, override vals for groups > 0
#                 if not (self.invariance == "configural" or f in self.partial_invariance.get("loadings_free_factors", [])) and g > 0:
#                     vals = numpyro.deterministic(f"lam_g{g}_f{f}_constrained", lam_list[0][idx, f])
#                 vals = vals.at[0].set(1.0)  # Marker fix AFTER override
#                 lam_g = lam_g.at[idx, f].set(vals)
#             lam_list.append(lam_g)
#         lam = jnp.stack(lam_list)

#         # --------------------------
#         # Cross-loadings (optional)
#         # --------------------------

#         if self.cross_loadings:
#             cross = numpyro.sample(
#                 "cross_loadings",
#                 dist.Normal(0.0, self.cross_loading_scale).expand([n_items, F]),
#             )
#             cross = jnp.where(self.primary_mask, 0.0, cross)
#             lam = lam + cross

#         # --------------------------
#         # Latent factor scores (eta)
#         # --------------------------

#         z = numpyro.sample(
#             "z",
#             dist.StudentT(self.df_latent, 0.0, 1.0).expand([n_obs, F]),
#         )

#         # Determine which covariance matrix to use per observation:
#         # If configural invariance OR any variances free, use group-specific covariance, else group 0
#         any_var_free = len(self.partial_invariance.get("variances_free_factors", [])) > 0
#         L_omega_used = (
#             L_omega_g[group] if (self.invariance == "configural" or any_var_free) else jnp.broadcast_to(L_omega_g[0], (n_obs, F, F))
#         )

#         eta = jnp.einsum("nf,nfh->nh", z, L_omega_used)

#         mu = jnp.einsum(
#             "nf,nif->ni",
#             eta + latent_means[group],
#             lam[group],
#         )

#         # --------------------------
#         # Likelihood
#         # --------------------------

#         lik = OrderedProbit(mu, thresholds[group])
#         numpyro.sample("obs", lik, obs=y)

#         # --------------------------
#         # Deterministic outputs for diagnostics
#         # --------------------------

#         numpyro.deterministic("log_likelihood", lik.log_prob(y).sum(axis=1))
#         numpyro.deterministic("loadings", lam)
#         numpyro.deterministic("thresholds", thresholds)
#         numpyro.deterministic("latent_means", latent_means)
#         numpyro.deterministic("factor_cov_group", cov_g)

#         std_g = jnp.sqrt(jnp.diagonal(cov_g, axis1=1, axis2=2))
#         corr_g = cov_g / (std_g[:, None, :] * std_g[:, :, None])
#         numpyro.deterministic("factor_corr_group", corr_g)

In [ ]:
# import warnings
# import jax.numpy as jnp
# from jax.nn import softplus
# import numpyro
# import numpyro.distributions as dist

# class BayesianOrdinalMGCFA:
#     def __init__(
#         self,
#         n_factors: int,
#         n_cats: int,
#         group: jnp.ndarray,
#         n_groups: int,
#         model_type: str = "cfa",          # "cfa" or "null"
#         item_to_factor_indices=None,
#         invariance: str = "metric",        # "configural", "metric", "scalar"
#         scalar_type: str | None = None,    # Optional, "relaxed" or "strict" ONLY if invariance == "scalar"
#         cross_loadings: bool = False,
#         loading_loc_prior: float = 0.6,
#         loading_scale_prior: float = 0.2,
#         cross_loading_scale: float = 0.05,
#         min_threshold_spacing: float = 0.05,
#         df_latent: float = 7.0,
#         partial_invariance: dict | None = None,
#         # ---------- ITEM CLUSTER EFFECTS ----------
#         use_cluster_effects: bool = False,
#         item_to_cluster_indices=None,
#         cluster_sd_prior: float = 0.3,
#     ):
#         self.n_factors = n_factors
#         self.n_cats = n_cats
#         self.group = group
#         self.n_groups = n_groups
#         self.model_type = model_type.lower()
#         self.invariance = invariance.lower()
#         self.cross_loadings = cross_loadings
#         self.loading_loc_prior = loading_loc_prior
#         self.loading_scale_prior = loading_scale_prior
#         self.cross_loading_scale = cross_loading_scale
#         self.min_threshold_spacing = min_threshold_spacing
#         self.df_latent = df_latent

#         assert self.model_type in ("null", "cfa")
#         assert self.invariance in ("configural", "metric", "scalar")

#         # Handle scalar_type only if invariance=="scalar"
#         if self.invariance == "scalar":
#             if scalar_type is None:
#                 self.scalar_type = "relaxed"
#             else:
#                 self.scalar_type = scalar_type.lower()
#                 assert self.scalar_type in ("relaxed", "strict")
#         else:
#             if scalar_type is not None:
#                 warnings.warn(
#                     f"scalar_type argument ignored because invariance='{self.invariance}' (not 'scalar')."
#                 )
#             self.scalar_type = None  # Disabled when not scalar

#         self.partial_invariance = partial_invariance or {
#             "loadings_free_factors": [],
#             "intercepts_free_factors": [],
#             "variances_free_factors": [],
#         }

#         self.item_to_factor_indices = item_to_factor_indices

#         # ---------- cluster bookkeeping ----------
#         self.use_cluster_effects = use_cluster_effects
#         self.item_to_cluster_indices = item_to_cluster_indices
#         self.cluster_sd_prior = cluster_sd_prior

#         if self.model_type == "null":
#             self.primary_mask = None
#             return

#         # --------------------------
#         # Item bookkeeping
#         # --------------------------
#         self.n_items = sum(len(idx) for idx in item_to_factor_indices)

#         mask = jnp.zeros((self.n_items, n_factors), dtype=bool)
#         for f, idx in enumerate(item_to_factor_indices):
#             mask = mask.at[idx, f].set(True)
#         self.primary_mask = mask

#         # --------------------------
#         # Item cluster index
#         # --------------------------
#         if self.use_cluster_effects:
#             assert item_to_cluster_indices is not None

#             cluster_index = jnp.zeros(self.n_items, dtype=int)
#             for c, idx in enumerate(item_to_cluster_indices):
#                 cluster_index = cluster_index.at[idx].set(c)

#             self.cluster_index = cluster_index
#             self.n_clusters = len(item_to_cluster_indices)

#     def model(self, data: jnp.ndarray):
#         n_obs, n_items = data.shape
#         y = data
#         group = self.group
#         G = self.n_groups
#         F = self.n_factors

#         # --------------------------
#         # Threshold locations (tau0)
#         # --------------------------
#         tau0 = numpyro.sample(
#             "tau0",
#             dist.Normal(0.0, 1.0).expand(
#                 [G, n_items] if self.invariance != "scalar" else [n_items]
#             ),
#         )
#         if self.invariance == "scalar":
#             tau0 = jnp.broadcast_to(tau0, (G, n_items))

#         # --------------------------
#         # Threshold spacings (delta)
#         # --------------------------
#         if self.invariance == "scalar" and self.scalar_type == "strict":
#             # strict scalar invariance: identical spacings
#             raw_delta = numpyro.sample(
#                 "raw_delta",
#                 dist.Normal(0.0, 1.0).expand([n_items, self.n_cats - 2]),
#             )
#             delta = softplus(raw_delta) + self.min_threshold_spacing
#             delta = jnp.broadcast_to(delta, (G, n_items, self.n_cats - 2))
#         else:
#             # relaxed scalar OR metric/configural
#             raw_delta = numpyro.sample(
#                 "raw_delta",
#                 dist.Normal(0.0, 1.0).expand([G, n_items, self.n_cats - 2]),
#             )
#             delta = softplus(raw_delta) + self.min_threshold_spacing

#         thresholds = jnp.cumsum(
#             jnp.concatenate([jnp.zeros((G, n_items, 1)), delta], axis=-1),
#             axis=-1,
#         )
#         thresholds = thresholds + tau0[..., None]
#         # identification
#         thresholds = thresholds.at[..., 0].set(0.0)
#         thresholds = jnp.sort(thresholds, axis=-1)

#         # --------------------------
#         # Latent means
#         # --------------------------
#         latent_means = jnp.zeros((G, F))
#         z_latent_means = numpyro.sample(
#             "z_latent_means",
#             dist.StudentT(self.df_latent, 0.0, 1.0).expand([G - 1, F]),
#         )
#         latent_means = latent_means.at[1:].set(z_latent_means)

#         # --------------------------
#         # Factor covariance
#         # --------------------------
#         L_corr = numpyro.sample(
#             "L_corr",
#             dist.LKJCholesky(dimension=F, concentration=2.0),
#         )

#         sigma_g = numpyro.sample(
#             "sigma_g",
#             dist.HalfNormal(0.5).expand([G, F]),
#         )

#         L_omega_g = L_corr * sigma_g[:, None, :]
#         cov_g = jnp.einsum("gij,gkj->gik", L_omega_g, L_omega_g)

#         # --------------------------
#         # Factor loadings
#         # --------------------------
#         lam_list = []
#         for g in range(G):
#             lam_g = jnp.zeros((n_items, F))
#             for f, idx in enumerate(self.item_to_factor_indices):
#                 vals = numpyro.sample(
#                     f"lam_g{g}_f{f}",
#                     dist.TruncatedNormal(
#                         loc=self.loading_loc_prior,
#                         scale=self.loading_scale_prior,
#                         low=0.0,
#                     ).expand([len(idx)]),
#                 )
#                 # marker item
#                 vals = vals.at[0].set(1.0)
#                 lam_g = lam_g.at[idx, f].set(vals)
#             lam_list.append(lam_g)

#         lam = jnp.stack(lam_list)

#         # --------------------------
#         # Latent scores
#         # --------------------------
#         z = numpyro.sample(
#             "z",
#             dist.StudentT(self.df_latent, 0.0, 1.0).expand([n_obs, F]),
#         )

#         eta = jnp.einsum("nf,nfh->nh", z, L_omega_g[group])

#         mu = jnp.einsum(
#             "nf,nif->ni",
#             eta + latent_means[group],
#             lam[group],
#         )

#         # --------------------------
#         # ITEM CLUSTER EFFECTS
#         # --------------------------
#         if self.use_cluster_effects:
#             cluster_sd = numpyro.sample(
#                 "cluster_sd",
#                 dist.HalfNormal(self.cluster_sd_prior),
#             )

#             cluster_effects = numpyro.sample(
#                 "cluster_effects",
#                 dist.Normal(0.0, cluster_sd).expand([G, self.n_clusters]),
#             )

#             mu = mu + cluster_effects[group][:, self.cluster_index]

#         # --------------------------
#         # Likelihood
#         # --------------------------
#         lik = OrderedProbit(mu, thresholds[group])
#         numpyro.sample("obs", lik, obs=y)

#         # --------------------------
#         # Diagnostics
#         # --------------------------
#         numpyro.deterministic("log_likelihood", lik.log_prob(y).sum(axis=1))
#         numpyro.deterministic("loadings", lam)
#         numpyro.deterministic("factor_cov_group", cov_g)

#         if self.use_cluster_effects:
#             numpyro.deterministic("cluster_effects", cluster_effects)

In [92]:
# # ============================================================
# # MCMC RUN LOOP
# # ============================================================

# Y = preobs_data_array
# item_indices = [
#     jnp.array([0, 1, 2]),    # Factor 1 items
#     jnp.array([3, 4, 5]),    # Factor 2 items
#     jnp.array([6, 7, 8]),    # Factor 3 items
#     jnp.array([9,10,11]),    # Factor 4 items
#     jnp.array([12,13,14])    # Factor 5 items
# ]

# partial_invariance = {
#     "loadings_free_factors": [2],       # Freeing the loadings of factor 3 "luck"
#     "intercepts_free_factors": [2],      # Freeing the thresholds of factor 3 "luck"
#     "variances_free_factors": [2],      # Freeing the variances of factor 3 "luck"
# }

# invariance_tests_idata = {}
# n_factors = 5
# n_cats = int(jnp.max(jnp.max(Y, axis=0)) + 1)

# # Basic sanity checks
# assert Y.min() >= 0
# assert Y.max() < n_cats

# for i, invariance in enumerate(["configural", "metric", "scalar"]):
#     print(f"\n=== {invariance.upper()} invariance ===")
    
#     model = BayesianOrdinalMGCFA(
#         n_factors=n_factors, n_cats=n_cats,
#         item_to_factor_indices=item_indices,
#         group=preobs_valence, n_groups=2,
#         invariance=invariance,
#         partial_invariance=partial_invariance,
#         model_type="cfa", cross_loadings=True
#     )
    
#     # FIXED: Clean PRNG + built-in init
#     rng_key = random.PRNGKey(42 + i * 10)
#     nuts = NUTS(model.model, init_strategy=init_to_median())  # Robust default
    
#     mcmc = MCMC(
#         nuts, num_warmup=2000, num_samples=4000, 
#         num_chains=4, progress_bar=True
#     )
    
#     # POSITIONAL ARG ONLY—no dict/kwargs!
#     mcmc.run(rng_key, Y)
    
#     invariance_tests_idata[invariance] = az.from_numpyro(mcmc, log_likelihood=True)
#     print(f"{invariance}: LOO={az.loo(invariance_tests_idata[invariance])['loo_i'].sum():.1f}")

# # Compare
# az.compare(invariance_tests_idata)  # ΔLOO invariance testing


=== CONFIGURAL invariance ===


/tmp/ipykernel_5259/2517371641.py:44: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(
warmup:   0%|                                            | 11/6000 [00:09<1:29:22,  1.12it/s, 1023 steps of size 6.69e-03. acc. prob=0.59]


KeyboardInterrupt: 

In [ ]:
import jax
import jax.numpy as jnp
from jax.scipy.special import log_ndtr
from numpyro.distributions import Distribution, constraints


def _logdiffexp(a, b):
    """Stable log(exp(a) - exp(b)) assuming a >= b."""
    return a + jnp.log1p(-jnp.exp(b - a))


class OrderedProbit(Distribution):
    arg_constraints = {"mu": constraints.real, "thresholds": constraints.ordered_vector}
    support = constraints.nonnegative_integer  # upper bound depends on n_thr

    def __init__(self, mu, thresholds, validate_args=None):
        self.mu = jnp.asarray(mu)
        thr = jnp.asarray(thresholds)

        if thr.ndim < 1:
            raise ValueError("thresholds must have at least 1 dimension (last dim = n_thr).")

        self.n_thr = int(thr.shape[-1])
        self.n_cats = self.n_thr + 1

        target_shape = self.mu.shape + (self.n_thr,)
        self.thresholds = jnp.broadcast_to(thr, target_shape)

        super().__init__(batch_shape=self.mu.shape, event_shape=(), validate_args=validate_args)

    def log_prob(self, value):
        y = jnp.asarray(value).astype(jnp.int32)
        in_range = (y >= 0) & (y < self.n_cats)
    
        t = self.thresholds - self.mu[..., None]
    
        neg_inf = jnp.full(self.mu.shape + (1,), -jnp.inf)
        pos_inf = jnp.full(self.mu.shape + (1,),  jnp.inf)
        t_full = jnp.concatenate([neg_inf, t, pos_inf], axis=-1)
    
        y_exp = y[..., None]
        lower = jnp.take_along_axis(t_full[..., :-1], y_exp, axis=-1)[..., 0]
        upper = jnp.take_along_axis(t_full[...,  1:], y_exp, axis=-1)[..., 0]
    
        ordered = upper >= lower
    
        log_cdf_u = log_ndtr(upper)
        log_cdf_l = log_ndtr(lower)
    
        # Numerical safety: enforce log_cdf_u >= log_cdf_l up to floating error,
        # without "fixing" invalid threshold order (that's already handled by `ordered`).
        # This only prevents tiny roundoff from causing exp(b-a) > 1.
        log_cdf_u_safe = jnp.maximum(log_cdf_u, log_cdf_l)
    
        log_p = _logdiffexp(log_cdf_u_safe, log_cdf_l)
    
        return jnp.where(in_range & ordered, log_p, -jnp.inf)

    def sample(self, key, sample_shape=()):
        """
        Returns integer categories with shape sample_shape + batch_shape.
        """
        shape = sample_shape + self.batch_shape
        eps = jax.random.normal(key, shape)
        z = self.mu + eps  # latent response, probit scale (residual var fixed to 1)

        # Count thresholds crossed: sum_k I[z > tau_k]
        # thresholds: batch_shape + (n_thr,)
        # z[..., None]: sample_shape + batch_shape + (1,)
        y = jnp.sum(z[..., None] > self.thresholds, axis=-1)

        return y.astype(jnp.int32)

In [95]:
# -*- coding: utf-8 -*-
from __future__ import annotations

from typing import Optional, Sequence, List, Tuple
import warnings

import numpy as np  # NEW: host-side validation
import jax.numpy as jnp
from jax.nn import softplus

import numpyro
import numpyro.distributions as dist

# from your_distributions import OrderedProbit


class BayesianOrdinalMGCFA:
    """
    Bayesian ordinal multi-group CFA using Ordered Probit (Normal CDF Φ).

    Notes:
    - Group 0 latent means fixed to 0.
    - Ordered probit implies latent-response residual variance fixed to 1.
    - Efficiency: samples only active/free deviation entries and scatters them in.
    """

    def __init__(
        self,
        n_factors: int,
        n_cats: int,
        group: jnp.ndarray,
        n_groups: int,
        model_type: str = "cfa",
        item_to_factor_indices: Optional[Sequence[jnp.ndarray]] = None,
        invariance: str = "metric",                 # configural, metric, scalar
        partial_invariance: Optional[dict] = None,
        cross_loadings: bool = False,
        # priors / hyperparams
        loading_loc_prior: float = 0.6,
        loading_scale_prior: float = 0.2,
        loading_prior: str = "truncnorm",           # truncnorm | lognormal
        cross_loading_scale: float = 0.05,
        min_threshold_spacing: float = 0.05,
        df_latent: float = 7.0,
        lkj_concentration: float = 4.0,
        thr_dev_scale_prior: float = 0.15,
        load_dev_scale_prior: float = 0.10,
        var_dev_scale_prior: float = 0.10,
        t0_dev_scale_prior: float = 0.5,
        equal_latent_sds_by_default: bool = True,
        latent_scale: str = "marker",               # marker | unitvar
        threshold_location: str = "fixed0",         # fixed0 | free
        store_deterministics: bool = True,
        debug_shapes: bool = False,
    ):
        if item_to_factor_indices is None:
            raise ValueError("item_to_factor_indices must be provided.")

        self.n_factors = int(n_factors)
        self.n_cats = int(n_cats)
        self.group = jnp.asarray(group).astype(jnp.int32)
        self.n_groups = int(n_groups)

        self.model_type = str(model_type).lower()
        self.invariance = str(invariance).lower()
        self.cross_loadings = bool(cross_loadings)

        self.loading_loc_prior = float(loading_loc_prior)
        self.loading_scale_prior = float(loading_scale_prior)
        self.loading_prior = str(loading_prior).lower()
        self.cross_loading_scale = float(cross_loading_scale)

        self.min_threshold_spacing = float(min_threshold_spacing)
        self.df_latent = float(df_latent)
        self.lkj_concentration = float(lkj_concentration)

        self.thr_dev_scale_prior = float(thr_dev_scale_prior)
        self.load_dev_scale_prior = float(load_dev_scale_prior)
        self.var_dev_scale_prior = float(var_dev_scale_prior)
        self.t0_dev_scale_prior = float(t0_dev_scale_prior)

        self.equal_latent_sds_by_default = bool(equal_latent_sds_by_default)
        self.latent_scale = str(latent_scale).lower()
        self.threshold_location = str(threshold_location).lower()

        self.store_deterministics = bool(store_deterministics)
        self.debug_shapes = bool(debug_shapes)

        if self.model_type not in ("null", "cfa"):
            raise ValueError("model_type must be 'null' or 'cfa'.")
        if self.invariance not in ("configural", "metric", "scalar"):
            raise ValueError("invariance must be one of: configural, metric, scalar.")
        if self.loading_prior not in ("truncnorm", "lognormal"):
            raise ValueError("loading_prior must be 'truncnorm' or 'lognormal'.")
        if self.lkj_concentration <= 0:
            raise ValueError("lkj_concentration must be > 0.")
        if self.latent_scale not in ("marker", "unitvar"):
            raise ValueError("latent_scale must be 'marker' or 'unitvar'.")
        if self.threshold_location not in ("fixed0", "free"):
            raise ValueError("threshold_location must be 'fixed0' or 'free'.")

        # indices as JAX arrays
        self.item_to_factor_indices = [jnp.asarray(ix, dtype=jnp.int32) for ix in item_to_factor_indices]
        self.n_items = sum(int(len(ix)) for ix in self.item_to_factor_indices)

        # primary mask [n_items,F]
        primary_mask = jnp.zeros((self.n_items, self.n_factors), dtype=bool)
        for f, idx in enumerate(self.item_to_factor_indices):
            primary_mask = primary_mask.at[idx, f].set(True)
        self.primary_mask = primary_mask

        # markers: first item per factor
        self.marker_items = jnp.array([int(ix[0]) for ix in self.item_to_factor_indices], dtype=jnp.int32)

        # non-marker primary mask
        marker_mask = jnp.zeros((self.n_items, self.n_factors), dtype=bool)
        for f in range(self.n_factors):
            marker_mask = marker_mask.at[int(self.marker_items[f]), f].set(True)
        self.nonmarker_primary_mask = self.primary_mask & (~marker_mask)

        # partial invariance factor-lists
        self.partial_invariance = partial_invariance or {
            "loadings_free_factors": [],
            "intercepts_free_factors": [],
            "variances_free_factors": [],
            "residuals_free_factors": [],
        }

        # masks for "free by factor"
        self._free_thr_items = self._build_free_threshold_item_mask()                  # [n_items]
        self._free_primary_loading_mask = self._build_free_primary_loading_mask()      # [n_items,F], excludes marker
        self._free_variance_mask = self._build_free_variance_mask()                    # [F]

        # indices for threshold-free items
        self._thr_free_item_idx = jnp.where(self._free_thr_items)[0].astype(jnp.int32)
        self._n_thr_free_items = int(self._thr_free_item_idx.shape[0])

        # indices for loading deviations (static for this model instance)
        self._lam_dev_rows_conf, self._lam_dev_cols_conf = self._where_mask(self.nonmarker_primary_mask)
        self._lam_dev_rows_rest, self._lam_dev_cols_rest = self._where_mask(self._free_primary_loading_mask)

        # indices for variance deviations (static)
        self._var_free_idx = jnp.where(self._free_variance_mask)[0].astype(jnp.int32)

        if self.cross_loadings:
            warnings.warn(
                "cross_loadings=True is often unstable with 3 indicators/factor and asymmetric groups."
            )
        if self.loading_prior == "lognormal" and self.loading_scale_prior > 0.25:
            warnings.warn(
                "LogNormal loading prior with loading_scale_prior > 0.25 can be heavy-tailed."
            )

    # ----------------------------
    # mask helpers
    # ----------------------------
    @staticmethod
    def _where_mask(mask_2d: jnp.ndarray) -> Tuple[jnp.ndarray, jnp.ndarray]:
        rows, cols = jnp.where(mask_2d)
        return rows.astype(jnp.int32), cols.astype(jnp.int32)

    def _build_free_threshold_item_mask(self) -> jnp.ndarray:
        free_factors: List[int] = list(self.partial_invariance.get("intercepts_free_factors", []))
        m = jnp.zeros((self.n_items,), dtype=bool)
        for f in free_factors:
            m = m.at[self.item_to_factor_indices[f]].set(True)
        return m

    def _build_free_primary_loading_mask(self) -> jnp.ndarray:
        free_factors: List[int] = list(self.partial_invariance.get("loadings_free_factors", []))
        m = jnp.zeros((self.n_items, self.n_factors), dtype=bool)
        for f in free_factors:
            idx = self.item_to_factor_indices[f]
            if len(idx) > 1:
                m = m.at[idx[1:], f].set(True)  # exclude marker
        return m & self.primary_mask

    def _build_free_variance_mask(self) -> jnp.ndarray:
        free_factors: List[int] = list(self.partial_invariance.get("variances_free_factors", []))
        m = jnp.zeros((self.n_factors,), dtype=bool)
        for f in free_factors:
            m = m.at[f].set(True)
        return m

    # ----------------------------
    # priors
    # ----------------------------
    def _sample_positive_loadings(self, name: str, shape):
        if self.loading_prior == "truncnorm":
            return numpyro.sample(
                name,
                dist.TruncatedNormal(
                    loc=self.loading_loc_prior,
                    scale=self.loading_scale_prior,
                    low=0.0,
                ).expand(shape),
            )
        mu = jnp.log(self.loading_loc_prior) - 0.5 * (self.loading_scale_prior ** 2)
        s = self.loading_scale_prior
        return numpyro.sample(name, dist.LogNormal(loc=mu, scale=s).expand(shape))

    # ----------------------------
    # model
    # ----------------------------
    def model(self, data: jnp.ndarray, predictive_group: Optional[jnp.ndarray] = None):
        # Host-side bounds check (safe under JIT/tracing)
        if predictive_group is not None:
            g_np = np.asarray(predictive_group)
            if g_np.size == 0:
                raise ValueError("predictive_group is empty.")
            if g_np.min() < 0 or g_np.max() >= self.n_groups:
                raise ValueError(
                    f"predictive_group values must be in [0, {self.n_groups - 1}], "
                    f"got range [{g_np.min()}, {g_np.max()}]."
                )

        y = jnp.asarray(data)
        n_obs, n_items = y.shape
        if n_items != self.n_items:
            raise ValueError(f"data has {n_items} items but model expects {self.n_items}.")

        g = self.group if predictive_group is None else jnp.asarray(predictive_group).astype(jnp.int32)
        G = self.n_groups
        F = self.n_factors
        K = self.n_cats
        n_thr = K - 1  # finite thresholds per item

        # ============================================================
        # THRESHOLDS
        # ============================================================
        if K < 2:
            raise ValueError("n_cats must be >= 2.")

        # K=2 binary: one threshold per item
        if K == 2:
            if self.threshold_location == "fixed0":
                t0_g = jnp.zeros((G, n_items))
            else:
                t0_base = numpyro.sample("t0_base", dist.Normal(0.0, 1.0).expand([n_items]))
                t0_g = jnp.broadcast_to(t0_base[None, :], (G, n_items))

                if G > 1:
                    t0_dev_active = (self.invariance != "scalar") or (self._n_thr_free_items > 0)
                    if t0_dev_active:
                        t0_dev_scale = numpyro.sample("t0_dev_scale", dist.HalfNormal(self.t0_dev_scale_prior))
                        if self.invariance == "scalar":
                            t0_z_free = numpyro.sample(
                                "t0_z_free",
                                dist.Normal(0.0, 1.0).expand([G - 1, self._n_thr_free_items]),
                            )
                            t0_g = t0_g.at[1:, self._thr_free_item_idx].add(t0_dev_scale * t0_z_free)
                        else:
                            t0_z = numpyro.sample("t0_z", dist.Normal(0.0, 1.0).expand([G - 1, n_items]))
                            t0_g = t0_g.at[1:, :].add(t0_dev_scale * t0_z)

            thresholds = t0_g[..., None]  # [G,n_items,1]

        else:
            # K>=3: K-2 positive spacings
            n_sp = n_thr - 1  # = K-2

            raw_spacings_base = numpyro.sample(
                "raw_spacings_base",
                dist.Normal(0.0, 1.0).expand([n_items, n_sp]),
            )
            raw_spacings_g = jnp.broadcast_to(raw_spacings_base, (G, n_items, n_sp))

            spacings_dev_active = False
            if G > 1:
                spacings_dev_active = (self.invariance != "scalar") or (self._n_thr_free_items > 0)

            if spacings_dev_active:
                dev_scale_sp = numpyro.sample("raw_spacings_dev_scale", dist.HalfNormal(self.thr_dev_scale_prior))
                if self.invariance == "scalar":
                    z_free = numpyro.sample(
                        "raw_spacings_z_free",
                        dist.Normal(0.0, 1.0).expand([G - 1, self._n_thr_free_items, n_sp]),
                    )
                    raw_spacings_g = raw_spacings_g.at[1:, self._thr_free_item_idx, :].add(dev_scale_sp * z_free)
                else:
                    z_all = numpyro.sample(
                        "raw_spacings_z",
                        dist.Normal(0.0, 1.0).expand([G - 1, n_items, n_sp]),
                    )
                    raw_spacings_g = raw_spacings_g.at[1:, :, :].add(dev_scale_sp * z_all)

            delta_g = softplus(raw_spacings_g) + self.min_threshold_spacing
            t_rest = jnp.cumsum(delta_g, axis=-1)  # [G,n_items,n_sp]

            # first threshold location
            if self.threshold_location == "fixed0":
                t0_g = jnp.zeros((G, n_items))
            else:
                t0_base = numpyro.sample("t0_base", dist.Normal(0.0, 1.0).expand([n_items]))
                t0_g = jnp.broadcast_to(t0_base[None, :], (G, n_items))

                t0_dev_active = False
                if G > 1:
                    t0_dev_active = (self.invariance != "scalar") or (self._n_thr_free_items > 0)

                if t0_dev_active:
                    t0_dev_scale = numpyro.sample("t0_dev_scale", dist.HalfNormal(self.t0_dev_scale_prior))
                    if self.invariance == "scalar":
                        t0_z_free = numpyro.sample(
                            "t0_z_free",
                            dist.Normal(0.0, 1.0).expand([G - 1, self._n_thr_free_items]),
                        )
                        t0_g = t0_g.at[1:, self._thr_free_item_idx].add(t0_dev_scale * t0_z_free)
                    else:
                        t0_z = numpyro.sample("t0_z", dist.Normal(0.0, 1.0).expand([G - 1, n_items]))
                        t0_g = t0_g.at[1:, :].add(t0_dev_scale * t0_z)

            thresholds = jnp.concatenate([t0_g[..., None], t0_g[..., None] + t_rest], axis=-1)  # [G,n_items,n_thr]

        # ============================================================
        # NULL model
        # ============================================================
        if self.model_type == "null":
            mu0 = jnp.zeros((n_obs, n_items))
            lik = OrderedProbit(mu0, thresholds[g])
            numpyro.sample("obs", lik, obs=y)
            log_prob_y = lik.log_prob(y)  # expected [n_obs, n_items]
            numpyro.deterministic("log_likelihood", log_prob_y.sum(axis=1))
            if self.store_deterministics:
                numpyro.deterministic("thresholds", thresholds)
            return

        # ============================================================
        # LATENT MEANS (group 0 reference)
        # ============================================================
        latent_means = jnp.zeros((G, F))
        if G > 1:
            z_latent_means = numpyro.sample(
                "z_latent_means",
                dist.StudentT(self.df_latent, 0.0, 1.0).expand([G - 1, F]),
            )
            latent_means = latent_means.at[1:].set(z_latent_means)

        # ============================================================
        # LATENT COVARIANCE: LKJ corr + factor SDs
        # ============================================================
        L_corr = numpyro.sample(
            "L_corr",
            dist.LKJCholesky(dimension=F, concentration=self.lkj_concentration),
        )  # [F,F]

        log_sigma_g = jnp.zeros((G, F))

        if self.latent_scale == "unitvar":
            # group 0 fixed to 1 => log_sigma_g[0]=0
            if G > 1:
                if self.invariance == "configural" or (not self.equal_latent_sds_by_default):
                    var_idx = jnp.arange(F, dtype=jnp.int32)
                else:
                    var_idx = self._var_free_idx
                n_var = int(var_idx.shape[0])
                if n_var > 0:
                    z_sig = numpyro.sample(
                        "log_sigma_z_free",
                        dist.Normal(0.0, 1.0).expand([G - 1, n_var]),
                    )
                    sig_dev_scale = numpyro.sample(
                        "log_sigma_dev_scale_free",
                        dist.HalfNormal(self.var_dev_scale_prior).expand([n_var]),
                    )
                    log_sigma_g = log_sigma_g.at[1:, var_idx].set(z_sig * sig_dev_scale[None, :])
        else:
            # marker: estimate baseline SDs for group 0, then deviations for groups 1..
            log_sigma_base = numpyro.sample(
                "log_sigma_base",
                dist.Normal(-0.5, 0.5).expand([F]),
            )
            log_sigma_g = log_sigma_g.at[0, :].set(log_sigma_base)

            if G > 1:
                if self.invariance == "configural" or (not self.equal_latent_sds_by_default):
                    var_idx = jnp.arange(F, dtype=jnp.int32)
                else:
                    var_idx = self._var_free_idx
                n_var = int(var_idx.shape[0])
                if n_var > 0:
                    z_sig = numpyro.sample(
                        "log_sigma_z_free",
                        dist.Normal(0.0, 1.0).expand([G - 1, n_var]),
                    )
                    sig_dev_scale = numpyro.sample(
                        "log_sigma_dev_scale_free",
                        dist.HalfNormal(self.var_dev_scale_prior).expand([n_var]),
                    )
                    log_sigma_g = log_sigma_g.at[1:, :].set(log_sigma_base[None, :])
                    log_sigma_g = log_sigma_g.at[1:, var_idx].add(z_sig * sig_dev_scale[None, :])
                else:
                    log_sigma_g = log_sigma_g.at[1:, :].set(log_sigma_base[None, :])

        sigma_g = jnp.exp(log_sigma_g)  # [G,F]
        # L_omega_g = diag(sigma_g) @ L_corr ; shape [G,F,F]
        L_omega_g = sigma_g[:, :, None] * L_corr[None, :, :]

        # ============================================================
        # LOADINGS: baseline (group 0) via marker method
        # ============================================================
        lam_base = jnp.zeros((n_items, F))
        for f, idx in enumerate(self.item_to_factor_indices):
            lam_base = lam_base.at[int(idx[0]), f].set(1.0)
            if len(idx) > 1:
                vals = self._sample_positive_loadings(
                    name=f"lam_base_f{f}_nonmarker",
                    shape=[len(idx) - 1],
                )
                lam_base = lam_base.at[idx[1:], f].set(vals)

        lam = jnp.broadcast_to(lam_base, (G, n_items, F))

        if self.invariance == "configural":
            free_rows, free_cols = self._lam_dev_rows_conf, self._lam_dev_cols_conf
        else:
            free_rows, free_cols = self._lam_dev_rows_rest, self._lam_dev_cols_rest
        n_free_lam = int(free_rows.shape[0])

        if G > 1 and n_free_lam > 0:
            lam_dev_scale = numpyro.sample(
                "lam_dev_scale",
                dist.HalfNormal(self.load_dev_scale_prior).expand([F]),
            )
            lam_z_free = numpyro.sample(
                "lam_z_free",
                dist.Normal(0.0, 1.0).expand([G - 1, n_free_lam]),
            )
            lam = lam.at[1:, free_rows, free_cols].add(lam_z_free * lam_dev_scale[None, free_cols])

        # Markers are excluded from free_rows/free_cols by construction (nonmarker_primary_mask).
        # The set(1.0) below is a defensive guarantee, not strictly necessary.
        for f in range(F):
            lam = lam.at[:, int(self.marker_items[f]), f].set(1.0)

        if self.cross_loadings:
            cross = numpyro.sample(
                "cross_loadings",
                dist.Normal(0.0, self.cross_loading_scale).expand([n_items, F]),
            )
            cross = jnp.where(self.primary_mask, 0.0, cross)
            lam = lam + cross[None, :, :]

        # ============================================================
        # LATENT SCORES eta and predictor mu
        # ============================================================
        z = numpyro.sample(
            "z",
            dist.StudentT(self.df_latent, 0.0, 1.0).expand([n_obs, F]),
        )

        L_obs = L_omega_g[g]  # [n_obs,F,F]
        # eta_n = L_{g(n)} @ z_n + mu_{g(n)}; Var(eta | g) = L L^T = Sigma_g
        eta = jnp.einsum("nij,nj->ni", L_obs, z) + latent_means[g]
        mu = jnp.einsum("nf,nif->ni", eta, lam[g])

        # ============================================================
        # Likelihood (always keep log_likelihood for LOO/WAIC)
        # ============================================================
        lik = OrderedProbit(mu, thresholds[g])
        numpyro.sample("obs", lik, obs=y)

        log_prob_y = lik.log_prob(y)  # expected [n_obs, n_items]
        numpyro.deterministic("log_likelihood", log_prob_y.sum(axis=1))

        if self.debug_shapes:
            if thresholds.shape != (G, self.n_items, self.n_cats - 1):
                raise ValueError(f"thresholds shape {thresholds.shape} unexpected")
            if L_omega_g.shape != (G, F, F):
                raise ValueError(f"L_omega_g shape {L_omega_g.shape} unexpected")
            if lam.shape != (G, self.n_items, F):
                raise ValueError(f"lam shape {lam.shape} unexpected")

        if self.store_deterministics:
            numpyro.deterministic("loadings", lam)
            numpyro.deterministic("thresholds", thresholds)
            numpyro.deterministic("latent_means", latent_means)

            cov_g = jnp.einsum("gij,gkj->gik", L_omega_g, L_omega_g)  # L L^T
            numpyro.deterministic("factor_cov_group", cov_g)

            std_g = jnp.sqrt(jnp.diagonal(cov_g, axis1=1, axis2=2))
            corr_g = cov_g / (std_g[:, None, :] * std_g[:, :, None])
            numpyro.deterministic("factor_corr_group", corr_g)

In [ ]:
# ============================================================
# FULL RUN + DIAGNOSTICS LOOP (updated for new BayesianOrdinalMGCFA)
# ============================================================

import jax.numpy as jnp
import jax.random as random

import arviz as az
from numpyro.infer import NUTS, MCMC, init_to_median

# ----------------------------
# INPUTS
# ----------------------------
Y = jnp.asarray(preobs_data_array)
group = jnp.asarray(preobs_valence).astype(jnp.int32)  # must be 0..G-1

item_indices = [
    jnp.array([0, 1, 2]),     # Factor 1
    jnp.array([3, 4, 5]),     # Factor 2
    jnp.array([6, 7, 8]),     # Factor 3 ("luck")
    jnp.array([9, 10, 11]),   # Factor 4
    jnp.array([12, 13, 14])   # Factor 5
]

partial_invariance = {
    "loadings_free_factors": [2],     # allow loading deviations (non-marker) for factor 3
    "intercepts_free_factors": [2],   # allow threshold deviations for factor 3 items under scalar invariance
    "variances_free_factors": [2],    # allow latent SD deviations for factor 3 under pooled-SD default
}

n_factors = 5
n_groups = 2

# ----------------------------
# SANITY CHECKS
# ----------------------------

# If your categories are 1..K, uncomment:
# Y = Y - 1

if Y.ndim != 2:
    raise ValueError("Y must be a 2D array of shape [n_obs, n_items].")

if int(Y.min()) < 0:
    raise ValueError("Y must be coded as integers 0..K-1 for OrderedProbit.")

n_cats = int(Y.max()) + 1

if int(group.min()) < 0 or int(group.max()) >= n_groups:
    raise ValueError(f"group must be coded 0..{n_groups-1}.")

n_obs, n_items = Y.shape
if n_items != 15:
    print(f"Warning: expected 15 items; got {n_items}.")

print(f"n_obs={n_obs}, n_items={n_items}, n_groups={n_groups}, n_cats={n_cats}")

# ----------------------------
# OPTIONAL: CATEGORY SPARSITY CHECK (CRITICAL FOR n=45 GROUP)
# ----------------------------
def ordinal_freq_table(y: jnp.ndarray, g: jnp.ndarray, n_cats: int):
    """
    Returns counts[g, item, cat] as a JAX array.
    """
    G = int(g.max()) + 1
    n_obs, n_items = y.shape
    counts = jnp.zeros((G, n_items, n_cats), dtype=jnp.int32)

    for gg in range(G):
        y_g = y[g == gg]  # [n_g, n_items]
        # count categories per item
        for c in range(n_cats):
            counts = counts.at[gg, :, c].set(jnp.sum(y_g == c, axis=0))
    return counts

counts = ordinal_freq_table(Y, group, n_cats)
small_g = 1  # assumes group==1 is the small group; change if needed
zeros_small = (counts[small_g] == 0).sum(axis=1)  # per item: number of zero-count categories
print("\nZero-count categories per item in group 1:", zeros_small.tolist())
print("Items with ANY zero-count category in group 1:", jnp.where(zeros_small > 0)[0].tolist())

# ----------------------------
# RUN SETTINGS
# ----------------------------
CROSS_LOADINGS = False          # strongly recommended for 3 items/factor + n=45
LKJ_CONC = 4.0                  # stronger regularization for small samples / few indicators
INIT_STRATEGY = init_to_median()

NUM_WARMUP = 2000
NUM_SAMPLES = 4000
NUM_CHAINS = 4

# ----------------------------
# FIT + COMPARE
# ----------------------------
invariance_tests_idata = {}
loos = {}

for i, invariance in enumerate(["configural", "metric", "scalar"]):
    print(f"\n=== {invariance.upper()} invariance ===")

    model = BayesianOrdinalMGCFA(
        n_factors=n_factors,
        n_cats=n_cats,
        item_to_factor_indices=item_indices,
        group=group,
        n_groups=n_groups,
        invariance=invariance,
        partial_invariance=partial_invariance,
        model_type="cfa",
        cross_loadings=CROSS_LOADINGS,

        # robust defaults / updated features
        lkj_concentration=LKJ_CONC,
        loading_prior="truncnorm",        # safest default
        equal_latent_sds_by_default=True,
        debug_shapes=False,
        # df_latent left as the class default unless you pass it here
        # df_latent=5.0,
    )

    rng_key = random.PRNGKey(42 + i * 10)
    nuts = NUTS(model.model, init_strategy=INIT_STRATEGY)

    mcmc = MCMC(
        nuts,
        num_warmup=NUM_WARMUP,
        num_samples=NUM_SAMPLES,
        num_chains=NUM_CHAINS,
        progress_bar=True,
    )
    mcmc.run(rng_key, Y)

    idata = az.from_numpyro(mcmc, log_likelihood=True)
    invariance_tests_idata[invariance] = idata

    # ----------------------------
    # BASIC SAMPLER DIAGNOSTICS
    # ----------------------------
    extra = mcmc.get_extra_fields()
    div = int(extra["diverging"].sum())
    print(f"Divergences: {div}")

    summ = az.summary(idata, var_names=["~log_likelihood"])
    max_rhat = float(summ["r_hat"].max())
    min_ess = float(summ["ess_bulk"].min())
    print(f"Max R-hat: {max_rhat:.3f} | Min ESS bulk: {min_ess:.1f}")

    # ----------------------------
    # LOO + PARETO-k
    # ----------------------------
    loo = az.loo(idata, pointwise=True)
    loos[invariance] = loo
    print(f"elpd_loo={loo.elpd_loo:.1f} | p_loo={loo.p_loo:.1f}")

    k = loo.pareto_k
    n_k07 = int((k > 0.7).sum().values)
    n_k10 = int((k > 1.0).sum().values)

    print(f"Pareto-k: #>0.7={n_k07}, #>1.0={n_k10}")

    # group-specific high-k counts
    k_vals = k.values
    g_vals = jnp.asarray(group).astype(int)
    n_k07_g0 = int(((k_vals > 0.7) & (jnp.array(g_vals) == 0)).sum())
    n_k07_g1 = int(((k_vals > 0.7) & (jnp.array(g_vals) == 1)).sum())
    print(f"High influence (k>0.7): group0={n_k07_g0}, group1={n_k07_g1}")

# ----------------------------
# MODEL COMPARISON
# ----------------------------
print("\n=== az.compare ===")
cmp = az.compare(invariance_tests_idata)
print(cmp)

# ----------------------------
# POST-SAMPLING POLISH (SCALAR MODEL)
# ----------------------------
if "scalar" in invariance_tests_idata:
    scalar_idata = invariance_tests_idata["scalar"]

    # A) Factor correlations by group (mean across chains/draws)
    if "factor_corr_group" in scalar_idata.posterior:
        f_corr = scalar_idata.posterior["factor_corr_group"].mean(dim=["chain", "draw"])

        # f_corr dims are typically (factor_corr_group_dim_0, factor_corr_group_dim_1, factor_corr_group_dim_2)
        # where dim_0 is group, dim_1 and dim_2 are factor indices.
        print("\nMean Factor Correlations (Group 0):")
        print(jnp.asarray(f_corr.isel(**{f_corr.dims[0]: 0})).round(2))

        print("\nMean Factor Correlations (Group 1):")
        print(jnp.asarray(f_corr.isel(**{f_corr.dims[0]: 1})).round(2))
    else:
        print("\nNo factor_corr_group found in posterior (check model deterministics).")

    # B) Inspect actual loading differences (group1 - group0) for the freed factor
    if "loadings" in scalar_idata.posterior:
        lam = scalar_idata.posterior["loadings"]  # dims: chain, draw, G, item, factor

        # Identify dim names robustly
        # Expected: (chain, draw, loadings_dim_0, loadings_dim_1, loadings_dim_2)
        # where loadings_dim_0=group, loadings_dim_1=item, loadings_dim_2=factor.
        d = lam.dims
        group_dim = d[2]
        item_dim = d[3]
        factor_dim = d[4]

        lam_diff_mean = (lam.sel(**{group_dim: 1}) - lam.sel(**{group_dim: 0})).mean(dim=["chain", "draw"])

        freed_factor = 2
        freed_items = jnp.array([6, 7, 8])  # factor 3 items in your mapping

        diff_vals = lam_diff_mean.isel(**{item_dim: freed_items, factor_dim: freed_factor}).values
        print(f"\nPosterior mean loading differences (group1 - group0) for factor {freed_factor}, items {freed_items.tolist()}:")
        print(jnp.round(jnp.asarray(diff_vals), 3))

        # Optional: forest plot for those differences
        # Make a derived xarray for plotting
        lam_diff_draws = (lam.sel(**{group_dim: 1}) - lam.sel(**{group_dim: 0})).sel(**{factor_dim: freed_factor})
        # lam_diff_draws dims: chain, draw, item
        # Select items and plot
        sel_diff = lam_diff_draws.isel(**{item_dim: freed_items})
        az.plot_forest(sel_diff, combined=True)
    else:
        print("\nNo loadings found in posterior (check model deterministics).")

In [ ]:
# In your loop, before az.compare:
# High-value diagnostic: check if NUTS is struggling with the irrelevant dimensions
# or the scale-location tradeoff.
for inv, idata in invariance_tests_idata.items():
    divergences = idata.sample_stats.diverging.sum().values
    energy_bfmi = az.bfmi(idata)
    print(f"[{inv}] Divergences: {divergences} | Min BFMI: {energy_bfmi.min():.2f}")

In [ ]:
save_dir = Path("results/mcmc_idata/partial_invariance/5_factor/new")
save_dir.mkdir(parents=True, exist_ok=True)

for name, idata in invariance_tests_idata.items():
    az.to_netcdf(idata, save_dir / f"{name}.nc")

In [ ]:
# ============================================================
# MCMC Null Model
# ============================================================

Y = preobs_data_array
item_indices = [
    jnp.array([0, 1, 2]),    # Factor 1 items
    jnp.array([3, 4, 5]),    # Factor 2
    jnp.array([6, 7, 8]),    # Factor 3
    jnp.array([9,10,11]),    # Factor 4
    jnp.array([12,13,14])    # Factor 5
]

n_factors = 5
n_cats = int(jnp.max(jnp.max(Y, axis=0)) + 1)

# Basic sanity checks
assert Y.min() >= 0
assert Y.max() < n_cats

print(f"\n=== Null model ===")
    
model = BayesianOrdinalMGCFA(
        n_factors=n_factors, n_cats=n_cats,
        # item_to_factor_indices=item_indices,
        group=preobs_valence, n_groups=2,
        # invariance=invariance,  # Pass directly
        model_type="null", cross_loadings=True
    )
    
# FIXED: Clean PRNG + built-in init
rng_key = random.PRNGKey(42 + i * 10)
nuts = NUTS(model.model, init_strategy=init_to_median())  # Robust default
    
mcmc = MCMC(
    nuts, num_warmup=2000, num_samples=4000, 
    num_chains=4, progress_bar=True
)
    
# POSITIONAL ARG ONLY—no dict/kwargs!
mcmc.run(rng_key, Y)
    
invariance_tests_idata["null"] = az.from_numpyro(mcmc, log_likelihood=True)
# print(f"null: LOO={az.loo(invariance_tests_idata["null"])['loo_i'].sum():.1f}")

# Compare
az.compare(invariance_tests_idata)  # ΔLOO invariance testing

In [ ]:
output_folder = Path("outputs/cfa/partial_invariance/5_factor/new")

os.makedirs(output_folder, exist_ok=True)

In [ ]:
load_dir_partial_invariance = Path("results/mcmc_idata/partial_invariance/5_factor/new")

partial_invariance_tests_idata = {
    path.stem: az.from_netcdf(path)
    for path in load_dir_partial_invariance.glob("*.nc")
}

In [ ]:
partial_invariance_tests_idata.keys()

In [ ]:
cfa_null_idata = invariance_tests_idata["null"]

In [ ]:
cfa_configural_idata = invariance_tests_idata["configural"]

In [ ]:
cfa_metric_idata = invariance_tests_idata["metric"]

In [ ]:
cfa_scalar_idata = partial_invariance_tests_idata["scalar"]

In [ ]:
az.compare(invariance_tests_idata)  # ΔLOO invariance testing

In [ ]:
load_dir_full_invariance = Path("results/mcmc_idata/invariance/5_factor/new")

full_invariance_tests_idata = {
    path.stem: az.from_netcdf(path)
    for path in load_dir_full_invariance.glob("*.nc")
}

In [ ]:
cfa_scalar_idata = full_invariance_tests_idata["scalar"]

In [ ]:
cfa_scalar_idata.posterior.data_vars


In [ ]:
import arviz as az
import numpy as np
import pandas as pd

latent_means = cfa_scalar_idata.posterior["latent_means"].stack(sample=("chain", "draw"))


In [ ]:
luck_idx = 2  # Factor 3
luck_means = latent_means.sel(latent_means_dim_1=luck_idx)

In [ ]:
rows = []

for g in luck_means.latent_means_dim_0.values:
    samples = luck_means.sel(latent_means_dim_0=g).values

    rows.append({
        "Group": f"Group {g}",
        "Mean": samples.mean(),
        "Median": np.median(samples),
        "95% HDI Lower": az.hdi(samples, 0.95)[0],
        "95% HDI Upper": az.hdi(samples, 0.95)[1],
    })

luck_mean_table = pd.DataFrame(rows)
luck_mean_table

In [ ]:
import numpy as np
import pandas as pd
import arviz as az

def extract_latent_means(
    idata,
    factor_names,
    group_names,
    hdi_prob=0.95,
):
    da = idata.posterior["latent_means"]

    # Convert to numpy: (chain, draw, group, factor)
    values = da.values

    C, D, G, F = values.shape

    if F != len(factor_names):
        raise ValueError(
            f"Model has {F} latent factors, but {len(factor_names)} names were provided."
        )

    if G != len(group_names):
        raise ValueError(
            f"Model has {G} groups, but {len(group_names)} names were provided."
        )

    rows = []

    for g in range(G):
        for f in range(F):
            samples = values[:, :, g, f].reshape(-1)

            hdi = az.hdi(samples, hdi_prob)

            rows.append({
                "Group": group_names[g],
                "Factor": factor_names[f],
                "Mean": float(samples.mean()),
                "Median": float(np.median(samples)),
                f"{int(hdi_prob*100)}% HDI Lower": float(hdi[0]),
                f"{int(hdi_prob*100)}% HDI Upper": float(hdi[1]),
            })

    return pd.DataFrame(rows)

In [ ]:
factor_names = ["Effort", "Ability", "Luck", "Task", "Interest"]
group_names = ["Failure", "Success"]

latent_means_df = extract_latent_means(
    idata=cfa_scalar_idata,
    factor_names=factor_names,
    group_names=group_names,
    hdi_prob=0.95,
)

latent_means_df

In [ ]:
loo = az.loo(cfa_scalar_idata, pointwise=True)

pareto_k = loo.pareto_k.values  # shape: (n_persons,)

# Summary
print("Max Pareto-k:", pareto_k.max())
print("Proportion > 0.7:", np.mean(pareto_k > 0.7))
print("Proportion > 1.0:", np.mean(pareto_k > 1.0))

In [ ]:
loo = az.loo(cfa_configural_idata, pointwise=True)

pareto_k = loo.pareto_k.values  # shape: (n_persons,)

# Summary
print("Max Pareto-k:", pareto_k.max())
print("Proportion > 0.7:", np.mean(pareto_k > 0.7))
print("Proportion > 1.0:", np.mean(pareto_k > 1.0))

In [ ]:
loo = az.loo(cfa_metric_idata, pointwise=True)

pareto_k = loo.pareto_k.values  # shape: (n_persons,)

# Summary
print("Max Pareto-k:", pareto_k.max())
print("Proportion > 0.7:", np.mean(pareto_k > 0.7))
print("Proportion > 1.0:", np.mean(pareto_k > 1.0))

In [ ]:
from numpyro.infer import Predictive

# mcmc: your fitted MCMC object
# model: your BayesianOrdinalMGCFA.model function
# data: your observed data array (n_obs x n_items)
# rng_key: PRNG key for sampling

rng_key = jax.random.PRNGKey(0)
predictive = Predictive(model.model, mcmc.get_samples(), return_sites=["obs"])

ppc_samples = predictive(rng_key, preobs_data_array)
# ppc_samples["obs"] shape: (n_samples, n_obs, n_items)

In [ ]:
idata = az.from_numpyro(mcmc)

ppc_samples = predictive(rng_key, preobs_data_array)
ppc_dataset = az.from_dict(posterior_predictive=ppc_samples).posterior_predictive

idata.add_groups(posterior_predictive=ppc_dataset)

az.plot_ppc(idata, group="posterior", kind="cumulative", num_pp_samples=100)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_ppc_ordinal_item(obs, ppc_samples, model, item_idx, num_pp_samples=100):
    """
    Plot observed category proportions vs posterior predictive category proportions
    for one ordinal item with uncertainty bands.

    Parameters:
    - obs: (n_obs, n_items) numpy array of observed zero-based ordinal data
    - ppc_samples: (n_samples, n_obs, n_items) numpy array of posterior predictive draws
    - model: your model object with attribute n_cats (number of categories)
    - item_idx: int, index of the item to plot
    - num_pp_samples: int, number of posterior predictive samples to plot for uncertainty
    """

    n_cats = model.n_cats
    n_obs = obs.shape[0]

    # Observed category proportions for the selected item
    obs_counts = np.bincount(obs[:, item_idx], minlength=n_cats) / n_obs

    # Subsample posterior predictive samples (if there are more than requested)
    ppc_subsamples = ppc_samples[:num_pp_samples]

    # Compute category proportions for each posterior predictive sample
    ppc_cat_props = np.array([
        np.bincount(sample[:, item_idx], minlength=n_cats) / n_obs
        for sample in ppc_subsamples
    ])

    # Compute mean and 90% credible interval for the posterior predictive category proportions
    ppc_mean = ppc_cat_props.mean(axis=0)
    ppc_lower = np.percentile(ppc_cat_props, 5, axis=0)
    ppc_upper = np.percentile(ppc_cat_props, 95, axis=0)

    # Plot setup
    categories = np.arange(n_cats)
    plt.figure(figsize=(8, 5))

    # Plot posterior predictive samples as thin gray lines for uncertainty
    for sample_props in ppc_cat_props:
        plt.plot(categories, sample_props, color='gray', alpha=0.15, lw=1)

    # Posterior predictive mean
    plt.plot(categories, ppc_mean, color='blue', lw=3, label='Posterior predictive mean')

    # 90% credible interval bands
    plt.fill_between(categories, ppc_lower, ppc_upper, color='blue', alpha=0.2, label='90% credible interval')

    # Observed data proportions as red points with stem lines
    plt.stem(categories, obs_counts, linefmt='r-', markerfmt='ro', basefmt='k-', label='Observed data')

    plt.xlabel("Category")
    plt.ylabel("Proportion")
    plt.title(f"Posterior Predictive Check for Item {item_idx}")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
plot_ppc_ordinal_item(preobs_data_array, ppc_samples_np, model, item_idx=0)
plot_ppc_ordinal_item(preobs_data_array, ppc_samples_np, model, item_idx=1)

In [ ]:
print("ppc_samples_np shape:", ppc_samples_np.shape)
print("ppc_samples_np min:", ppc_samples_np.min())
print("ppc_samples_np max:", ppc_samples_np.max())
print("ppc_samples_np dtype:", ppc_samples_np.dtype)

In [ ]:
print(az.summary(cfa_configural_idata, var_names=["loadings", "thresholds", "latent_means", "L_omega"]))

In [ ]:
az.summary(cfa_scalar_idata)

In [ ]:
az.plot_trace(cfa_scalar_idata)

In [ ]:
az.plot_rank(cfa_scalar_idata)

In [ ]:
cfa_metric_idata = az.from_numpyro(mcmc)

idata = az.from_numpyro(mcmc)
print(az.summary(idata, var_names=["loadings", "thresholds", "latent_means", "L_omega"]))

az.plot_trace(idata)
az.plot_energy(idata)
az.plot_rank(idata)

In [ ]:
cfa_scalar_idata = az.from_numpyro(mcmc)

idata = az.from_numpyro(mcmc)
print(az.summary(idata, var_names=["loadings", "thresholds", "latent_means", "L_omega"]))

az.plot_trace(idata)
az.plot_energy(idata)
az.plot_rank(idata)

In [ ]:
loo_result = az.loo(trace, scale="deviance")
print(loo_result.pareto_k)

In [ ]:
# Get the k-hat values (shape: number of observations)
khat_values = az.loo(cfa_configural_idata).pareto_k.values

# Find indices of problematic points
problematic_points = (khat_values > 0.7).nonzero()[0]

print(f"Number of problematic points: {len(problematic_points)}")
print(f"Indices of problematic points: {problematic_points}")

# Optionally print k-hat values for these points
for idx in problematic_points:
    print(f"Obs {idx}: k-hat = {khat_values[idx]:.3f}")


In [ ]:
import numpy as np
import arviz as az
import pandas as pd

def inspect_loadings(
    idata,
    n_items,
    n_factors,
    n_groups,
    item_names,
    invariance_type="configural",
    partial_invariance=None,
    standardize=True,
    return_intervals=True,
    hdi_prob=0.95,
    compute_reliability=True,
    compact_ci=True,
    summary_stat="mean",
):
    """
    Inspect Bayesian ordinal CFA loadings with posterior standardization and omega reliability.

    Parameters:
    - idata: InferenceData from numpyro sampling
    - n_items, n_factors, n_groups: ints describing data structure
    - item_names: list of item names
    - invariance_type: "configural", "metric", or "scalar"
    - partial_invariance: dict with keys like 'loadings_free_factors' listing factors with free loadings
    - standardize: whether to standardize loadings
    - return_intervals: whether to return HDI intervals
    - hdi_prob: HDI probability
    - compute_reliability: whether to compute omega reliability
    - compact_ci: if True, combines mean and CI into one string in output table
    - summary_stat: "mean" or "median"
    """

    if partial_invariance is None:
        partial_invariance = {
            "loadings_free_factors": [],
            "intercepts_free_factors": [],
            "variances_free_factors": [],
            "residuals_free_factors": [],
        }

    # 1. Extract loadings posterior and stack chains & draws
    lam_xr = idata.posterior["loadings"].stack(sample=("chain", "draw"))
    dims_lam = lam_xr.dims
    sizes_lam = lam_xr.sizes

    group_dim_lam = next(d for d in dims_lam if sizes_lam[d] == n_groups)
    item_dim = next(d for d in dims_lam if sizes_lam[d] == n_items)
    factor_dim_lam = next(d for d in dims_lam if sizes_lam[d] == n_factors)
    sample_dim = next(d for d in dims_lam if d == "sample")

    lam_xr = lam_xr.transpose(sample_dim, group_dim_lam, item_dim, factor_dim_lam)
    lam = lam_xr.values  # shape (samples, groups, items, factors)

    # 2. Extract factor covariance posterior and stack chains & draws
    factor_cov_xr = idata.posterior["factor_cov_group"].stack(sample=("chain", "draw"))
    dims_cov = factor_cov_xr.dims
    sizes_cov = factor_cov_xr.sizes

    group_dim_cov = next(d for d in dims_cov if sizes_cov[d] == n_groups)
    factor_dim_cov1 = next(d for d in dims_cov if sizes_cov[d] == n_factors)
    factor_dim_cov2 = next(d for d in dims_cov if sizes_cov[d] == n_factors and d != factor_dim_cov1)
    sample_dim_cov = next(d for d in dims_cov if d == "sample")

    factor_cov_xr = factor_cov_xr.transpose(sample_dim_cov, group_dim_cov, factor_dim_cov1, factor_dim_cov2)
    factor_cov = factor_cov_xr.values  # shape (samples, groups, factors, factors)

    n_samples, G, I, F = lam.shape

    # 3. Compute factor variances and residual variances per sample, group, item
    factor_vars = np.diagonal(factor_cov, axis1=2, axis2=3)  # (samples, groups, factors)

    residual_vars = np.zeros((n_samples, G, I))
    for s in range(n_samples):
        for g in range(G):
            communality = np.sum(lam[s, g, :, :]**2 * factor_vars[s, g, :], axis=1)
            residual_vars[s, g, :] = np.clip(1 - communality, 1e-6, None)

    # 4. Standardize loadings if requested
    if standardize:
        lam_std = np.zeros_like(lam)
        for s in range(n_samples):
            for g in range(G):
                for i in range(I):
                    for f in range(F):
                        denom = np.sqrt(
                            factor_vars[s, g, f] *
                            (factor_vars[s, g, f] * lam[s, g, i, f]**2 + residual_vars[s, g, i])
                        )
                        lam_std[s, g, i, f] = lam[s, g, i, f] / denom if denom > 0 else 0
    else:
        lam_std = lam

    # 5. Posterior summary for loadings
    if summary_stat == "mean":
        lam_center = lam_std.mean(axis=0)
    elif summary_stat == "median":
        lam_center = np.median(lam_std, axis=0)
    else:
        raise ValueError("summary_stat must be 'mean' or 'median'")

    # 6. Compute HDI intervals if requested
    if return_intervals:
        hdi_lo = np.zeros((G, I, F))
        hdi_hi = np.zeros((G, I, F))
        for g in range(G):
            for i in range(I):
                for f in range(F):
                    hdi = az.hdi(lam_std[:, g, i, f], hdi_prob)
                    hdi_lo[g, i, f] = hdi[0]
                    hdi_hi[g, i, f] = hdi[1]

    def fmt(mu, lo, hi):
        return f"{mu:.3f} [{lo:.3f}, {hi:.3f}]"

    # 7. Build loadings tables
    shared = invariance_type in ("metric", "scalar")
    loadings_tables = {}

    if shared:
        # If partial invariance on loadings, show separate loadings for free factors
        free_factors = partial_invariance.get("loadings_free_factors", [])
        if free_factors:
            for g in range(G):
                data = {}
                for f in range(F):
                    col = f"Factor {f+1}"
                    if return_intervals and compact_ci:
                        col += f" ({int(hdi_prob*100)}% CI)"
                        data[col] = [
                            fmt(lam_center[g, i, f], hdi_lo[g, i, f], hdi_hi[g, i, f])
                            for i in range(I)
                        ]
                    else:
                        data[col] = lam_center[g, :, f]
                loadings_tables[f"Group {g} (partial invariance)"] = pd.DataFrame(data, index=item_names)
        else:
            # Fully shared loadings table
            data = {}
            for f in range(F):
                col = f"Factor {f+1}"
                if return_intervals and compact_ci:
                    col += f" ({int(hdi_prob*100)}% CI)"
                    data[col] = [
                        fmt(lam_center[0, i, f], hdi_lo[0, i, f], hdi_hi[0, i, f])
                        for i in range(I)
                    ]
                else:
                    data[col] = lam_center[0, :, f]
            loadings_tables["Shared loadings"] = pd.DataFrame(data, index=item_names)

    else:
        # Configural invariance: separate tables for each group
        for g in range(G):
            data = {}
            for f in range(F):
                col = f"Factor {f+1}"
                if return_intervals and compact_ci:
                    col += f" ({int(hdi_prob*100)}% CI)"
                    data[col] = [
                        fmt(lam_center[g, i, f], hdi_lo[g, i, f], hdi_hi[g, i, f])
                        for i in range(I)
                    ]
                else:
                    data[col] = lam_center[g, :, f]
            loadings_tables[f"Group {g}"] = pd.DataFrame(data, index=item_names)

    # 8. Compute omega reliability if requested
    reliability_tables = None
    if compute_reliability:

        def omega_func(lam_sample, factor_var_sample, resid_var_sample):
            omega_per_factor = []
            for f in range(F):
                lam_f = lam_sample[:, f]
                phi_f = factor_var_sample[f]
                psi = resid_var_sample
                numerator = (lam_f.sum()) ** 2 * phi_f
                denominator = numerator + np.sum(lam_f**2 * phi_f * psi + phi_f * psi)
                omega_val = numerator / denominator if denominator > 0 else np.nan
                omega_per_factor.append(omega_val)
            return omega_per_factor

        reliability_tables = {}
        if shared and free_factors:
            # Omega per group (partial invariance)
            for g in range(G):
                omegas = np.zeros((n_samples, F))
                for s in range(n_samples):
                    omegas[s, :] = omega_func(lam[s, g], factor_vars[s, g], residual_vars[s, g])
                omega_mean = np.nanmean(omegas, axis=0)
                omega_median = np.nanmedian(omegas, axis=0)
                omega_hdi = np.array([az.hdi(omegas[:, f], hdi_prob) for f in range(F)])
                rel_df = pd.DataFrame(
                    {
                        "Omega Mean": omega_mean,
                        "Omega Median": omega_median,
                        f"{int(hdi_prob*100)}% HDI Lower": omega_hdi[:, 0],
                        f"{int(hdi_prob*100)}% HDI Upper": omega_hdi[:, 1],
                    },
                    index=[f"Factor {f+1}" for f in range(F)],
                )
                reliability_tables[f"Group {g} (partial invariance)"] = rel_df
        elif shared:
            # Shared omega
            omegas = np.zeros((n_samples, F))
            for s in range(n_samples):
                omegas[s, :] = omega_func(lam[s, 0], factor_vars[s, 0], residual_vars[s, 0])
            omega_mean = np.nanmean(omegas, axis=0)
            omega_median = np.nanmedian(omegas, axis=0)
            omega_hdi = np.array([az.hdi(omegas[:, f], hdi_prob) for f in range(F)])
            rel_df = pd.DataFrame(
                {
                    "Omega Mean": omega_mean,
                    "Omega Median": omega_median,
                    f"{int(hdi_prob*100)}% HDI Lower": omega_hdi[:, 0],
                    f"{int(hdi_prob*100)}% HDI Upper": omega_hdi[:, 1],
                },
                index=[f"Factor {f+1}" for f in range(F)],
            )
            reliability_tables["Shared reliability"] = rel_df
        else:
            # Per group omega
            for g in range(G):
                omegas = np.zeros((n_samples, F))
                for s in range(n_samples):
                    omegas[s, :] = omega_func(lam[s, g], factor_vars[s, g], residual_vars[s, g])
                omega_mean = np.nanmean(omegas, axis=0)
                omega_median = np.nanmedian(omegas, axis=0)
                omega_hdi = np.array([az.hdi(omegas[:, f], hdi_prob) for f in range(F)])
                rel_df = pd.DataFrame(
                    {
                        "Omega Mean": omega_mean,
                        "Omega Median": omega_median,
                        f"{int(hdi_prob*100)}% HDI Lower": omega_hdi[:, 0],
                        f"{int(hdi_prob*100)}% HDI Upper": omega_hdi[:, 1],
                    },
                    index=[f"Factor {f+1}" for f in range(F)],
                )
                reliability_tables[f"Group {g}"] = rel_df

    return loadings_tables, reliability_tables

In [ ]:
item_cols = ["eff_1", "eff_2", "eff_3", "ab_1", "ab_2", "ab_3", "luck_1", "luck_2", "luck_3", "task_1", "task_2", "task_3", "int_1", "int_2", "int_3"]

In [ ]:
loadings, reliability = inspect_loadings(
    cfa_scalar_idata,
    n_items=15,
    n_factors=5,
    n_groups=2,
    item_names=item_cols,
    invariance_type="scalar",
    standardize=False,
    return_intervals=True,
    hdi_prob=0.95,
    compute_reliability=True,
    compact_ci=True,
    summary_stat="mean",
    partial_invariance=partial_invariance,
)

In [ ]:
loadings

In [ ]:
reliability

In [ ]:
diagnostics_scalar_df = diagnostics_by_param_family(cfa_scalar_idata, rhat_threshold=1.01, rhat_acceptable=1.05)

print(diagnostics_scalar_df)

In [ ]:
filename = output_folder_success / f"diagnostics_factors_{n_factors}.tex"
    
latex_body = diagnostics_df.to_latex(index=False, escape=True, float_format="{:.2f}".format)

sideways = f"""
    \\begin{{sidewaystable}}[htbp]
    \\centering
    \\caption{{Diagnostics summary for {n_factors}-factor model (Success)}}
    {latex_body}
    \\end{{sidewaystable}}
    """
        
with open(filename, "w") as f:
    f.write(sideways)
        
print(f"Saved diagnostics table for {n_factors} factors to {filename}")
print("\n")

In [ ]:
def factor_correlation_table(
    idata,
    summary="mean",
    include_ci=True,
    ci_prob=0.95,
    factor_names=None,
    by_group=True,
    group_names=None,
):
    """
    Create tables of factor correlations from Bayesian (multi-group) CFA models.
    Supports configural, metric, scalar, and partial invariance models.

    Returns
    -------
    dict[str, pd.DataFrame] or pd.DataFrame
        One table per group if by_group=True, otherwise a pooled table.
    """

    da = idata.posterior["factor_corr_group"]
    dims = da.dims
    values = da.values

    # --------------------------------------------------
    # Identify dimensions
    # --------------------------------------------------
    has_group = any("group" in d for d in dims)
    has_chain = "chain" in dims and "draw" in dims
    has_sample = "sample" in dims

    # --------------------------------------------------
    # Reshape to (group, F, F, S)
    # --------------------------------------------------
    if has_chain:
        # Stack chain + draw → sample
        sample_shape = da.sizes["chain"] * da.sizes["draw"]
        values = values.reshape(sample_shape, *values.shape[2:])

        # Move sample to last axis
        values = np.moveaxis(values, 0, -1)

    elif has_sample:
        sample_axis = dims.index("sample")
        values = np.moveaxis(values, sample_axis, -1)

    else:
        raise ValueError(f"Unsupported dimensions: {dims}")

    # Now values is either:
    # (F, F, S) or (group, F, F, S)

    if values.ndim == 3:
        # Single-group → add dummy group axis
        values = values[None, ...]  # (1, F, F, S)

    G, F, _, S = values.shape

    # --------------------------------------------------
    # Naming
    # --------------------------------------------------
    if factor_names is None:
        factor_names = [f"Factor {i+1}" for i in range(F)]

    if group_names is None:
        group_names = [f"Group {g}" for g in range(G)]

    # --------------------------------------------------
    # Build tables
    # --------------------------------------------------
    tables = {}

    for g in range(G):
        rows = []

        for i in range(F):
            for j in range(i + 1, F):
                samples = values[g, i, j, :]

                if summary == "mean":
                    point = samples.mean()
                elif summary == "median":
                    point = np.median(samples)
                else:
                    raise ValueError("summary must be 'mean' or 'median'")

                row = {
                    "factor_1": factor_names[i],
                    "factor_2": factor_names[j],
                    "r": float(point),
                }

                if include_ci:
                    alpha = (1 - ci_prob) / 2
                    row["r_lower"] = float(np.quantile(samples, alpha))
                    row["r_upper"] = float(np.quantile(samples, 1 - alpha))

                rows.append(row)

        tables[group_names[g]] = pd.DataFrame(rows)

    # --------------------------------------------------
    # Return
    # --------------------------------------------------
    if by_group:
        return tables
    else:
        # Pool across groups
        pooled = pd.concat(
            tables.values(),
            keys=tables.keys(),
            names=["group"]
        ).reset_index(level="group")
        return pooled

In [ ]:
corr_tables = factor_correlation_table(
    cfa_scalar_idata,
    summary="mean",
    include_ci=True,
    factor_names=["Effort", "Ability", "Luck", "Task", "Interest"],
    by_group=True
)

In [ ]:
corr_tables["Group 0"]

In [ ]:
corr_tables["Group 1"]

In [13]:
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from numpyro.distributions import constraints
from jax.nn import softplus
from typing import Optional, Sequence


class BayesianMultigroupMultivariateCFA:
    def __init__(
        self,
        n_factors: int,
        n_cats: int,
        group: Optional[jnp.ndarray] = None,
        n_groups: int = 1,
        model_type: str = "cfa",  # "cfa" or "null"
        item_to_factor_indices: Optional[Sequence[jnp.ndarray]] = None,
        invariance: Optional[str] = "metric",  # "configural", "metric", "scalar"
        cross_loadings: bool = False,
        loading_loc_prior: float = 0.6,
        loading_scale_prior: float = 0.2,
        cross_loading_scale: float = 0.05,
        min_threshold_spacing: float = 0.05,
        df_latent: float = 7.0,
        partial_invariance: Optional[dict] = None,
        allowed_crossloading_mask: Optional[jnp.ndarray] = None,  # NEW: mask for allowed cross-loadings
    ):
        self.n_factors = n_factors
        self.n_cats = n_cats
        self.item_to_factor_indices = item_to_factor_indices
        self.n_groups = n_groups
        self.model_type = model_type.lower()
        self.invariance = invariance.lower() if invariance is not None else None
        self.cross_loadings = cross_loadings
        self.loading_loc_prior = loading_loc_prior
        self.loading_scale_prior = loading_scale_prior
        self.cross_loading_scale = cross_loading_scale
        self.min_threshold_spacing = min_threshold_spacing
        self.df_latent = df_latent
        self.partial_invariance = partial_invariance or {
            "loadings_free_factors": [],
            "intercepts_free_factors": [],
            "variances_free_factors": [],
            "residuals_free_factors": [],  # Placeholder, not implemented
        }

        # Handle group vector for single-group case
        if self.n_groups == 1:
            if group is None:
                raise ValueError(
                    "For single group, you must provide a group vector of zeros."
                )
            else:
                # verify group is all zeros
                if not jnp.all(group == 0):
                    raise ValueError(
                        "For single group, group vector must be all zeros."
                    )
        else:
            if group is None:
                raise ValueError("Group vector must be provided for multigroup model.")

        self.group = group

        # NULL model shortcut
        if self.model_type == "null":
            self.primary_mask = None
            return

        assert self.model_type in ("null", "cfa")
        assert self.invariance in ("configural", "metric", "scalar")

        n_items = sum(len(idx) for idx in item_to_factor_indices)
        mask = jnp.zeros((n_items, n_factors), dtype=bool)
        for f, idx in enumerate(item_to_factor_indices):
            mask = mask.at[idx, f].set(True)
        self.primary_mask = mask
        self.n_items = n_items

        # If cross_loadings enabled, set allowed mask
        if self.cross_loadings:
            if allowed_crossloading_mask is None:
                # Default: allow cross-loadings on all non-primary positions
                self.allowed_crossloading_mask = ~self.primary_mask
            else:
                # Use provided mask, but ensure primary loadings are excluded
                self.allowed_crossloading_mask = allowed_crossloading_mask & (~self.primary_mask)
        else:
            self.allowed_crossloading_mask = None

    def model(self, data: jnp.ndarray):
        n_obs, n_items = data.shape
        y = data
        group = self.group
        G = self.n_groups
        F = self.n_factors

        # --------------------------
        # Thresholds (Intercepts)
        # --------------------------

        tau0 = numpyro.sample(
            "tau0",
            dist.Normal(0.0, 1.0).expand(
                [G, n_items] if self.invariance != "scalar" else [n_items]
            ),
        )
        if self.invariance == "scalar":
            tau0 = jnp.broadcast_to(tau0, (G, n_items))

        intercepts_free_factors = self.partial_invariance.get("intercepts_free_factors", [])
        free_intercept_items = jnp.zeros(n_items, dtype=bool)
        for f in intercepts_free_factors:
            free_intercept_items = free_intercept_items.at[self.item_to_factor_indices[f]].set(True)

        raw_delta = numpyro.sample(
            "raw_delta",
            dist.Normal(0.0, 1.0).expand([G, n_items, self.n_cats - 2]),
        )
        delta = softplus(raw_delta) + self.min_threshold_spacing

        thresholds = jnp.cumsum(
            jnp.concatenate([jnp.zeros((G, n_items, 1)), delta], axis=-1),
            axis=-1,
        )
        thresholds = thresholds + tau0[..., None]

        thresholds = thresholds.at[..., 0].set(0.0)

        for f in range(F):
            if f not in intercepts_free_factors:
                idxs = self.item_to_factor_indices[f]
                thresholds = thresholds.at[1:, idxs, :].set(thresholds[0, idxs, :])

        thresholds = jnp.sort(thresholds, axis=-1)

        if self.model_type == "null":
            mu = jnp.zeros((n_obs, n_items))
            lik = OrderedProbit(mu, thresholds[group])
            numpyro.sample("obs", lik, obs=y)
            numpyro.deterministic("log_likelihood", lik.log_prob(y).sum(axis=1))
            return

        latent_means = jnp.zeros((G, F))
        z_latent_means = numpyro.sample(
            "z_latent_means",
            dist.StudentT(self.df_latent, 0.0, 1.0).expand([G - 1, F]),
        )
        latent_means = latent_means.at[1:].set(z_latent_means)

        L_corr = numpyro.sample(
            "L_corr",
            dist.LKJCholesky(dimension=F, concentration=2.0),
        )

        sigma_g = numpyro.sample(
            "sigma_g",
            dist.HalfNormal(0.5).expand([G, F]),
        )

        variances_free_factors = self.partial_invariance.get("variances_free_factors", [])
        for f in range(F):
            if f not in variances_free_factors:
                sigma_g = sigma_g.at[1:, f].set(sigma_g[0, f])

        L_omega_g = L_corr * sigma_g[:, None, :]
        cov_g = jnp.einsum("gij,gkj->gik", L_omega_g, L_omega_g)

        lam_list = []
        for g in range(G):
            lam_g = jnp.zeros((n_items, F))
            for f, idx in enumerate(self.item_to_factor_indices):
                vals = numpyro.sample(
                    f"lam_g{g}_f{f}",
                    dist.TruncatedNormal(
                        loc=self.loading_loc_prior,
                        scale=self.loading_scale_prior,
                        low=0.0,
                    ).expand([len(idx)]),
                )
                if not (self.invariance == "configural" or f in self.partial_invariance.get("loadings_free_factors", [])) and g > 0:
                    vals = numpyro.deterministic(f"lam_g{g}_f{f}_constrained", lam_list[0][idx, f])
                vals = vals.at[0].set(1.0)
                lam_g = lam_g.at[idx, f].set(vals)
            lam_list.append(lam_g)
        lam = jnp.stack(lam_list)

        if self.cross_loadings:
            cross = numpyro.sample(
                "cross_loadings",
                dist.Normal(0.0, self.cross_loading_scale).expand([n_items, F]),
            )
            cross = cross * self.allowed_crossloading_mask
            lam = lam + cross

        z = numpyro.sample(
            "z",
            dist.StudentT(self.df_latent, 0.0, 1.0).expand([n_obs, F]),
        )

        any_var_free = len(self.partial_invariance.get("variances_free_factors", [])) > 0
        L_omega_used = (
            L_omega_g[group] if (self.invariance == "configural" or any_var_free) else jnp.broadcast_to(L_omega_g[0], (n_obs, F, F))
        )

        eta = jnp.einsum("nf,nfh->nh", z, L_omega_used)

        mu = jnp.einsum(
            "nf,nif->ni",
            eta + latent_means[group],
            lam[group],
        )

        lik = OrderedProbit(mu, thresholds[group])
        numpyro.sample("obs", lik, obs=y)

        numpyro.deterministic("log_likelihood", lik.log_prob(y).sum(axis=1))
        numpyro.deterministic("loadings", lam)
        numpyro.deterministic("thresholds", thresholds)
        numpyro.deterministic("latent_means", latent_means)
        numpyro.deterministic("factor_cov_group", cov_g)

        std_g = jnp.sqrt(jnp.diagonal(cov_g, axis1=1, axis2=2))
        corr_g = cov_g / (std_g[:, None, :] * std_g[:, :, None])
        numpyro.deterministic("factor_corr_group", corr_g)

In [ ]:
# Attribution construct: 5 factors, items per factor (indices)
attribution_item_indices = [
    jnp.array([0,1,2]),     # Factor 1 items
    jnp.array([3,4,5]),     # Factor 2 items
    jnp.array([6,7,8]),     # Factor 3 items
    jnp.array([9,10]),      # Factor 4 items
    jnp.array([11,12,13]),  # Factor 5 items
]

# Self-efficacy construct: 3 factors
selfefficacy_item_indices = [
    jnp.array([14,15,16]),  # Factor 1 items
    jnp.array([17,18,19]),  # Factor 2 items
    jnp.array([20,21]),     # Factor 3 items
]

# Combine them all for joint model
all_item_indices = attribution_item_indices + selfefficacy_item_indices


In [ ]:
import jax.numpy as jnp
import numpy as np
import numpyro
from numpyro.infer import MCMC, NUTS

# Assume the class BayesianMultigroupMultivariateCFA is defined as above and imported here.
# Also you need the OrderedProbit likelihood implemented or imported.

# ---- Step 1: Define items to factor indices for the two constructs ----
# Attributions: 15 items, 5 factors (each factor has 3 items)
attr_factors = [jnp.array([0,1,2]), jnp.array([3,4,5]), jnp.array([6,7,8]), jnp.array([9,10,11]), jnp.array([12,13,14])]

# Self-efficacy: 9 items, 3 factors (each factor has 3 items)
selfeff_factors = [jnp.array([15,16,17]), jnp.array([18,19,20]), jnp.array([21,22,23])]

# Combine all
all_factors = attr_factors + selfeff_factors  # total factors = 8

n_factors = len(all_factors)  # 8 factors
n_items = sum(len(f) for f in all_factors)  # 24 items total

# ---- Step 2: Create allowed cross-loading mask ----
# We'll allow cross-loadings only within constructs, never across

# Initialize mask as all False
allowed_crossloading_mask = jnp.zeros((n_items, n_factors), dtype=bool)

# Allow crossloadings within Attributions (factors 0-4)
for f in range(5):
    items = all_factors[f]
    # For each item in this factor, allow cross-loading on any of the 5 attribution factors
    for item in items:
        allowed_crossloading_mask = allowed_crossloading_mask.at[item, 0:5].set(True)

# Allow crossloadings within Self-efficacy (factors 5-7)
for f in range(5, 8):
    items = all_factors[f]
    for item in items:
        allowed_crossloading_mask = allowed_crossloading_mask.at[item, 5:8].set(True)

# But the primary loading positions will be zeroed out in model code automatically

# ---- Step 3: Create dummy ordinal data ----
np.random.seed(123)
n_obs = 100
n_cats = 5  # assume 5-category ordinal responses

# Random ordinal data (for demo only)
data = np.random.randint(0, n_cats, size=(n_obs, n_items))
data = jnp.array(data)

# Group vector (single group)
group = jnp.zeros(n_obs, dtype=int)
n_groups = 1

# ---- Step 4: Instantiate model ----
model = BayesianMultigroupMultivariateCFA(
    n_factors=n_factors,
    n_cats=n_cats,
    group=group,
    n_groups=n_groups,
    item_to_factor_indices=all_factors,
    invariance="scalar",
    cross_loadings=True,
    allowed_crossloading_mask=allowed_crossloading_mask,
)

# ---- Step 5: Run MCMC ----
# Note: You need to implement OrderedProbit distribution or substitute with your preferred likelihood.

nuts_kernel = NUTS(model.model)
mcmc = MCMC(nuts_kernel, num_warmup=500, num_samples=1000)
mcmc.run(numpyro.PRNGKey(0), data)

# ---- Step 6: Summarize results ----
print(mcmc.print_summary())


## Backup: Old Code

In [ ]:
# # ============================================================
# # Bayesian Multi-Group Ordinal CFA
# # ============================================================

# class BayesianOrdinalMGCFA:
#     def __init__(
#         self,
#         n_factors: int,
#         n_cats: int,
#         group: jnp.ndarray,
#         n_groups: int,
#         model_type: str = "cfa",          # "cfa" or "null"
#         item_to_factor_indices: Optional[Sequence[jnp.ndarray]] = None,        
#         invariance: Optional[str] = "metric",  # "configural", "metric", "scalar"
#         cross_loadings: bool = False,
#         loading_loc_prior: float = 0.6,
#         loading_scale_prior: float = 0.2,
#         cross_loading_scale: float = 0.05,
#         min_threshold_spacing: float = 0.05,
#         df_latent: float = 7.0,
#     ):
#         self.n_factors = n_factors
#         self.n_cats = n_cats
#         self.item_to_factor_indices = item_to_factor_indices
#         self.group = group
#         self.n_groups = n_groups
#         self.model_type = model_type.lower()
#         self.invariance = invariance.lower() if invariance is not None else None
#         self.cross_loadings = cross_loadings
#         self.loading_loc_prior = loading_loc_prior
#         self.loading_scale_prior = loading_scale_prior
#         self.cross_loading_scale = cross_loading_scale
#         self.min_threshold_spacing = min_threshold_spacing
#         self.df_latent = df_latent

#         # NULL
#         if self.model_type == "null":
#             self.primary_mask = None
#             return
        
#         assert self.model_type in ("null", "cfa")
#         assert self.invariance in ("configural", "metric", "scalar")

#         # CFA
#         n_items = sum(len(idx) for idx in item_to_factor_indices)
#         mask = jnp.zeros((n_items, n_factors), dtype=bool)
#         for f, idx in enumerate(item_to_factor_indices):
#             mask = mask.at[idx, f].set(True)
#         self.primary_mask = mask

#     # --------------------------------------------------------
#     # Model
#     # --------------------------------------------------------

#     def model(self, data: jnp.ndarray):
#         n_obs, n_items = data.shape
#         y = data
#         group = self.group

#         G = self.n_groups
#         F = self.n_factors


#         # ----------------------------------------------------
#         # Thresholds (ordinal identification)
#         # ----------------------------------------------------

#         tau0 = numpyro.sample(
#             "tau0",
#             dist.Normal(0.0, 1.0).expand(
#                 [G, n_items] if self.invariance != "scalar" else [n_items]
#             ),
#         )

#         if self.invariance == "scalar":
#             tau0 = jnp.broadcast_to(tau0, (G, n_items))

#         raw_delta = numpyro.sample(
#             "raw_delta",
#             dist.Normal(0.0, 1.0).expand([G, n_items, self.n_cats - 2]),
#         )

#         delta = softplus(raw_delta) + self.min_threshold_spacing

#         thresholds = jnp.cumsum(
#             jnp.concatenate([jnp.zeros((G, n_items, 1)), delta], axis=-1),
#             axis=-1,
#         )

#         thresholds = thresholds + tau0[..., None]
#         thresholds = thresholds.at[..., 0].set(0.0)  # GLOBAL identification

#         # ----------------------------------------------------
#         # Null model (no factors)
#         # ----------------------------------------------------

#         if self.model_type == "null":
#             mu = jnp.zeros((n_obs, n_items))
#             lik = OrderedProbit(mu, thresholds[group])
#             numpyro.sample("obs", lik, obs=y)
#             numpyro.deterministic("log_likelihood", lik.log_prob(y).sum(axis=1))
#             return
            
#         # ----------------------------------------------------
#         # Latent means (group differences)
#         # ----------------------------------------------------

#         latent_means = jnp.zeros((G, F))
#         z_latent_means = numpyro.sample(
#             "z_latent_means",
#             dist.StudentT(self.df_latent, 0.0, 1.0).expand([G - 1, F]),
#         )
#         latent_means = latent_means.at[1:].set(z_latent_means)

#         # ----------------------------------------------------
#         # Factor covariance (identified)
#         # ----------------------------------------------------

#         L_corr = numpyro.sample(
#             "L_corr",
#             dist.LKJCholesky(dimension=F, concentration=2.0),
#         )

#         sigma_g = numpyro.sample(
#             "sigma_g",
#             dist.HalfNormal(0.5).expand([G, F]),
#         )
#         sigma_g = sigma_g.at[0].set(jnp.ones(F))

#         L_omega_g = L_corr * sigma_g[:, None, :]
#         cov_g = jnp.einsum("gij,gkj->gik", L_omega_g, L_omega_g)

#         # ----------------------------------------------------
#         # Factor loadings
#         # ----------------------------------------------------

#         def sample_loadings(name):
#             lam = jnp.zeros((n_items, F))
#             for f, idx in enumerate(self.item_to_factor_indices):
#                 vals = numpyro.sample(
#                     f"{name}_primary_f{f}",
#                     dist.TruncatedNormal(
#                         loc=self.loading_loc_prior,
#                         scale=self.loading_scale_prior,
#                         low=0.0,
#                     ).expand([idx.shape[0]]),
#                 )
#                 vals = vals.at[0].set(1.0)
#                 lam = lam.at[idx, f].set(vals)

#             if self.cross_loadings:
#                 cross = numpyro.sample(
#                     f"{name}_cross",
#                     dist.Normal(0.0, self.cross_loading_scale)
#                     .expand([n_items, F]),
#                 )
#                 cross = jnp.where(self.primary_mask, 0.0, cross)
#                 lam = lam + cross

#             return lam

#         if self.invariance == "configural":
#             lam = jnp.stack(
#                 [sample_loadings(f"lam_g{g}") for g in range(G)]
#             )
#         else:
#             lam_shared = sample_loadings("lam")
#             lam = jnp.broadcast_to(lam_shared, (G, n_items, F))

#         # ----------------------------------------------------
#         # Latent scores (robust)
#         # ----------------------------------------------------

#         z = numpyro.sample(
#             "z",
#             dist.StudentT(self.df_latent, 0.0, 1.0).expand([n_obs, F]),
#         )

#         L_omega_used = (
#             L_omega_g[group]
#             if self.invariance == "configural"
#             else jnp.broadcast_to(L_omega_g[0], (n_obs, F, F))
#         )

#         eta = jnp.einsum("nf,nfh->nh", z, L_omega_used)

#         mu = jnp.einsum(
#             "nf,nif->ni",
#             eta + latent_means[group],
#             lam[group],
#         )

#         # ----------------------------------------------------
#         # Likelihood
#         # ----------------------------------------------------

#         lik = OrderedProbit(mu, thresholds[group])
#         numpyro.sample("obs", lik, obs=y)

#         numpyro.deterministic(
#             "log_likelihood",
#             lik.log_prob(y).sum(axis=1),
#         )

#         numpyro.deterministic("loadings", lam)
#         numpyro.deterministic("thresholds", thresholds)
#         numpyro.deterministic("latent_means", latent_means)
#         numpyro.deterministic("factor_cov_group", cov_g)

#         std_g = jnp.sqrt(jnp.diagonal(cov_g, axis1=1, axis2=2))
#         corr_g = cov_g / (std_g[:, None, :] * std_g[:, :, None])
#         numpyro.deterministic("factor_corr_group", corr_g)